In [ ]:
import pandas as pd
import numpy as np

# Path to raw data folder (relative to notebooks/)
raw_path = "../data/raw/"

# Load the 5 small tables (leave chartevents.csv.gz and labevents.csv.gz untouched — multi-GB)
admissions = pd.read_csv(raw_path + "admissions.csv.gz", compression="gzip")
diagnoses_icd = pd.read_csv(raw_path + "diagnoses_icd.csv.gz", compression="gzip")
icustays = pd.read_csv(raw_path + "icustays.csv.gz", compression="gzip")
patients = pd.read_csv(raw_path + "patients.csv.gz", compression="gzip")
d_icd_diagnoses = pd.read_csv(raw_path + "d_icd_diagnoses.csv.gz", compression="gzip")

tables = {
    "admissions": admissions,
    "diagnoses_icd": diagnoses_icd,
    "icustays": icustays,
    "patients": patients,
    "d_icd_diagnoses": d_icd_diagnoses,
}

for name, df in tables.items():
    print(f"--- {name} ---")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print()

for name, df in tables.items():
    print(f"--- {name} head ---")
    display(df.head())
    print()

In [ ]:
# Search d_icd_diagnoses for candidate cancer and delirium codes

# --- Cancer / malignancy search terms ---
cancer_terms = ['malignant neoplasm', 'cancer', 'carcinoma', 'leukemia', 'lymphoma']
cancer_pattern = '|'.join(cancer_terms)

cancer_candidates = d_icd_diagnoses[
    d_icd_diagnoses['long_title'].str.contains(cancer_pattern, case=False, na=False, regex=True)
].copy()

cancer_candidates = cancer_candidates.sort_values(['icd_version', 'icd_code']).reset_index(drop=True)

print(f"Cancer/malignancy candidates found: {len(cancer_candidates)}")
print(cancer_candidates[['icd_code', 'icd_version', 'long_title']].to_string(index=False))
print()

# --- Delirium search terms (excluding encephalopathy) ---
delirium_terms = ['delirium', 'acute confusional state']
delirium_pattern = '|'.join(delirium_terms)

delirium_candidates = d_icd_diagnoses[
    d_icd_diagnoses['long_title'].str.contains(delirium_pattern, case=False, na=False, regex=True)
].copy()

delirium_candidates = delirium_candidates.sort_values(['icd_version', 'icd_code']).reset_index(drop=True)

print(f"Delirium candidates found: {len(delirium_candidates)}")
print(delirium_candidates[['icd_code', 'icd_version', 'long_title']].to_string(index=False))
print()

# --- Encephalopathy search, kept separate for manual review ---
encephalopathy_candidates = d_icd_diagnoses[
    d_icd_diagnoses['long_title'].str.contains('encephalopathy', case=False, na=False, regex=True)
].copy()

encephalopathy_candidates = encephalopathy_candidates.sort_values(['icd_version', 'icd_code']).reset_index(drop=True)

print(f"Encephalopathy candidates found (NOT merged into delirium list): {len(encephalopathy_candidates)}")
print(encephalopathy_candidates[['icd_code', 'icd_version', 'long_title']].to_string(index=False))

In [ ]:
import re

# ============================================================
# STEP 1: Build final CANCER code list programmatically
# ============================================================

def icd9_category(code):
    """Extract the 3-digit numeric category from an ICD-9 code string, if numeric."""
    code = str(code)
    if code[:3].isdigit():
        return int(code[:3])
    return None

def icd10_cancer_match(code):
    """True if code starts with 'C' followed by exactly two digits 00-96."""
    code = str(code)
    m = re.match(r'^C(\d{2})', code)
    if m:
        num = int(m.group(1))
        return 0 <= num <= 96
    return False

exclude_terms = [
    'in remission', 'personal history', 'family history', 'screening',
    'genetic susceptibility', 'follow-up', 'prophylactic',
    'complicating pregnancy', 'complicating childbirth', 'complicating the puerperium'
]
exclude_pattern = '|'.join(exclude_terms)

explicit_exclude_codes = {'79582', 'R971', 'R9721', 'D701'}

def is_carcinoma_in_situ(row):
    code = str(row['icd_code'])
    if row['icd_version'] == 9:
        cat = icd9_category(code)
        return cat is not None and 230 <= cat <= 234
    elif row['icd_version'] == 10:
        return code[:2] == 'D0'
    return False

# --- Range-based inclusion mask ---
def cancer_range_match(row):
    code = str(row['icd_code'])
    if row['icd_version'] == 9:
        cat = icd9_category(code)
        if cat is None:
            return False
        return (140 <= cat <= 172) or (174 <= cat <= 189) or (190 <= cat <= 209)
    elif row['icd_version'] == 10:
        return icd10_cancer_match(code)
    return False

cancer_candidates_work = cancer_candidates.copy()
cancer_candidates_work['icd_code'] = cancer_candidates_work['icd_code'].astype(str)

range_mask = cancer_candidates_work.apply(cancer_range_match, axis=1)
term_exclude_mask = cancer_candidates_work['long_title'].str.contains(exclude_pattern, case=False, na=False, regex=True)
explicit_exclude_mask = cancer_candidates_work['icd_code'].isin(explicit_exclude_codes)
cis_exclude_mask = cancer_candidates_work.apply(is_carcinoma_in_situ, axis=1)

final_include_mask = range_mask & ~term_exclude_mask & ~explicit_exclude_mask & ~cis_exclude_mask

cancer_final = cancer_candidates_work[final_include_mask].copy()
cancer_excluded = cancer_candidates_work[~final_include_mask].copy()

cancer_final_codes = set(cancer_final['icd_code'])

print(f"Final cancer codes KEPT: {len(cancer_final)}")
print(f"Final cancer codes EXCLUDED: {len(cancer_excluded)}")
print()

# ============================================================
# STEP 2: Build final DELIRIUM code list (exact, hardcoded)
# ============================================================

delirium_final_codes = {'F05', '2930', '2931'}
print(f"Final delirium codes kept: {sorted(delirium_final_codes)} (count: {len(delirium_final_codes)})")
print()

# ============================================================
# STEP 3: Flag hadm_id for cancer and delirium using diagnoses_icd
# ============================================================

diagnoses_icd_work = diagnoses_icd.copy()
diagnoses_icd_work['icd_code'] = diagnoses_icd_work['icd_code'].astype(str).str.strip()

diagnoses_icd_work['is_cancer'] = diagnoses_icd_work['icd_code'].isin(cancer_final_codes)
diagnoses_icd_work['is_delirium'] = diagnoses_icd_work['icd_code'].isin(delirium_final_codes)

hadm_flags = diagnoses_icd_work.groupby('hadm_id').agg(
    subject_id=('subject_id', 'first'),
    cancer=('is_cancer', 'any'),
    delirium=('is_delirium', 'any')
).reset_index()

# ============================================================
# STEP 4: Keep only hadm_id where BOTH cancer and delirium are True
# ============================================================

cohort_hadm = hadm_flags[(hadm_flags['cancer'] == True) & (hadm_flags['delirium'] == True)]

print(f"Cancer codes kept in final list: {len(cancer_final)}")
print(f"Cancer codes excluded: {len(cancer_excluded)}")
print(f"Delirium codes kept in final list: {len(delirium_final_codes)}")
print()
print(f"Unique subject_id count in combined cohort: {cohort_hadm['subject_id'].nunique()}")
print(f"Unique hadm_id count in combined cohort: {cohort_hadm['hadm_id'].nunique()}")

In [ ]:
# ============================================================
# STEP 1: Merge cancer+delirium hadm_id cohort with icustays and patients
# ============================================================

# Keep only the cancer+delirium-positive hadm_ids from the previous step
cohort_hadm_ids = cohort_hadm[['subject_id', 'hadm_id']].copy()

# Merge with icustays to bring in stay_id, intime, outtime, etc.
merged = cohort_hadm_ids.merge(
    icustays[['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los']],
    on=['subject_id', 'hadm_id'],
    how='inner'
)

merged['intime'] = pd.to_datetime(merged['intime'])
merged['outtime'] = pd.to_datetime(merged['outtime'])

print(f"Rows after merging cancer+delirium hadm_ids with icustays: {len(merged)}")
print(f"Unique subject_id at this point: {merged['subject_id'].nunique()}")
print()

# ============================================================
# STEP 2: Keep only the earliest ICU stay per subject_id (first ICU stay only)
# ============================================================

merged_sorted = merged.sort_values(['subject_id', 'intime'])
first_stay = merged_sorted.groupby('subject_id', as_index=False).first()

print(f"Count after first-ICU-stay-only filter: {len(first_stay)}")
print(f"Unique subject_id after first-ICU-stay-only filter: {first_stay['subject_id'].nunique()}")
print()

# ============================================================
# STEP 3: Merge with patients, compute age at ICU admission, filter age >= 18
# ============================================================

first_stay = first_stay.merge(
    patients[['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']],
    on='subject_id',
    how='left'
)

first_stay['age_at_admission'] = first_stay['anchor_age'] + (first_stay['intime'].dt.year - first_stay['anchor_year'])

cohort_step1 = first_stay[first_stay['age_at_admission'] >= 18].copy()

print(f"Count after age >= 18 filter: {len(cohort_step1)}")
print(f"Unique subject_id after age >= 18 filter: {cohort_step1['subject_id'].nunique()}")
print()

# ============================================================
# STEP 4: Save cohort_step1.csv with the required columns
# ============================================================

output_cols = ['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'age_at_admission']
cohort_step1_out = cohort_step1[output_cols].copy()

cohort_step1_out.to_csv('../data/processed/cohort_step1.csv', index=False)

print(f"Saved cohort_step1.csv with {len(cohort_step1_out)} rows")
print(cohort_step1_out.head())

In [ ]:
import pandas as pd

raw_path = "../data/raw/"

d_items = pd.read_csv(raw_path + "d_items.csv.gz", compression="gzip")

print(f"d_items shape: {d_items.shape}")
print(f"d_items columns: {d_items.columns.tolist()}")
print()

cam_icu_items = d_items[
    d_items['label'].str.contains('CAM-ICU', case=False, na=False) |
    d_items['label'].str.contains('confusion assessment', case=False, na=False)
].copy()

print(f"CAM-ICU related items found: {len(cam_icu_items)}")
print(cam_icu_items.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
import re

raw_path = "../data/raw/"

# ============================================================
# 1. RELOAD EVERYTHING FRESH FROM RAW FILES
# ============================================================

d_icd_diagnoses = pd.read_csv(raw_path + "d_icd_diagnoses.csv.gz", compression="gzip")
diagnoses_icd   = pd.read_csv(raw_path + "diagnoses_icd.csv.gz", compression="gzip")
icustays        = pd.read_csv(raw_path + "icustays.csv.gz", compression="gzip")
patients        = pd.read_csv(raw_path + "patients.csv.gz", compression="gzip")
d_items         = pd.read_csv(raw_path + "d_items.csv.gz", compression="gzip")

diagnoses_icd['icd_code'] = diagnoses_icd['icd_code'].astype(str).str.strip()
d_icd_diagnoses['icd_code'] = d_icd_diagnoses['icd_code'].astype(str).str.strip()
icustays['intime'] = pd.to_datetime(icustays['intime'])
icustays['outtime'] = pd.to_datetime(icustays['outtime'])

print("Raw tables reloaded:")
print(f"  d_icd_diagnoses: {d_icd_diagnoses.shape}")
print(f"  diagnoses_icd:   {diagnoses_icd.shape}")
print(f"  icustays:        {icustays.shape}")
print(f"  patients:        {patients.shape}")
print(f"  d_items:         {d_items.shape}")
print()

# ============================================================
# 2. FINAL CANCER ICD CODE LIST
# ============================================================

def icd9_category(code):
    code = str(code)
    return int(code[:3]) if code[:3].isdigit() else None

def icd10_cancer_match(code):
    m = re.match(r'^C(\d{2})', str(code))
    return 0 <= int(m.group(1)) <= 96 if m else False

def cancer_range_match(row):
    code = str(row['icd_code'])
    if row['icd_version'] == 9:
        cat = icd9_category(code)
        if cat is None:
            return False
        return (140 <= cat <= 172) or (174 <= cat <= 189) or (190 <= cat <= 209)
    elif row['icd_version'] == 10:
        return icd10_cancer_match(code)
    return False

def is_carcinoma_in_situ(row):
    code = str(row['icd_code'])
    if row['icd_version'] == 9:
        cat = icd9_category(code)
        return cat is not None and 230 <= cat <= 234
    elif row['icd_version'] == 10:
        return code[:2] == 'D0'
    return False

exclude_terms = [
    'in remission', 'personal history', 'family history', 'screening',
    'genetic susceptibility', 'follow-up', 'prophylactic',
    'complicating pregnancy', 'complicating childbirth', 'complicating the puerperium'
]
exclude_pattern = '|'.join(exclude_terms)
explicit_exclude_codes = {'79582', 'R971', 'R9721', 'D701'}

range_mask = d_icd_diagnoses.apply(cancer_range_match, axis=1)
term_exclude_mask = d_icd_diagnoses['long_title'].str.contains(exclude_pattern, case=False, na=False, regex=True)
explicit_exclude_mask = d_icd_diagnoses['icd_code'].isin(explicit_exclude_codes)
cis_exclude_mask = d_icd_diagnoses.apply(is_carcinoma_in_situ, axis=1)

cancer_include_mask = range_mask & ~term_exclude_mask & ~explicit_exclude_mask & ~cis_exclude_mask
cancer_final_codes = set(d_icd_diagnoses.loc[cancer_include_mask, 'icd_code'])

print(f"Final cancer ICD codes: {len(cancer_final_codes)}")

# ============================================================
# 3. FINAL DELIRIUM ICD CODE LIST (exact, hardcoded)
# ============================================================

delirium_final_codes = {'F05', '2930', '2931'}
print(f"Final delirium ICD codes: {sorted(delirium_final_codes)}")
print()

# ============================================================
# CANCER-POSITIVE hadm_id (needed before chartevents chunking)
# ============================================================

diagnoses_icd['is_cancer'] = diagnoses_icd['icd_code'].isin(cancer_final_codes)
diagnoses_icd['is_delirium_icd'] = diagnoses_icd['icd_code'].isin(delirium_final_codes)

hadm_flags = diagnoses_icd.groupby('hadm_id').agg(
    subject_id=('subject_id', 'first'),
    cancer=('is_cancer', 'any'),
    delirium_icd=('is_delirium_icd', 'any')
).reset_index()

cancer_positive_hadm_ids = set(hadm_flags.loc[hadm_flags['cancer'], 'hadm_id'])
delirium_icd_only_hadm_ids = set(hadm_flags.loc[hadm_flags['delirium_icd'], 'hadm_id'])

print(f"Cancer-positive hadm_id count: {len(cancer_positive_hadm_ids)}")
print(f"Delirium-positive hadm_id count (ICD-only): {len(delirium_icd_only_hadm_ids)}")
print()

# ============================================================
# 4. CAM-ICU ITEMS FROM d_items (confirmed: 11 items, all Text, all chartevents)
# ============================================================

cam_icu_items = d_items[
    d_items['label'].str.contains('CAM-ICU', case=False, na=False) |
    d_items['label'].str.contains('confusion assessment', case=False, na=False)
].copy()

cam_icu_itemids = set(cam_icu_items['itemid'])
print(f"CAM-ICU itemids being pulled from chartevents: {sorted(cam_icu_itemids)}")
print()

# ============================================================
# PULL chartevents ONLY for CAM-ICU itemids + cancer-positive hadm_ids
# (chunked, never loaded fully) — NO value interpretation yet
# ============================================================

chunk_size = 1_000_000
cam_icu_rows = []
usecols = ['subject_id', 'hadm_id', 'itemid', 'charttime', 'value', 'valuenum']

for chunk in pd.read_csv(
    raw_path + "chartevents.csv.gz",
    compression="gzip",
    usecols=usecols,
    chunksize=chunk_size
):
    filtered = chunk[
        chunk['itemid'].isin(cam_icu_itemids) &
        chunk['hadm_id'].isin(cancer_positive_hadm_ids)
    ]
    if len(filtered) > 0:
        cam_icu_rows.append(filtered)

cam_icu_events = pd.concat(cam_icu_rows, ignore_index=True) if cam_icu_rows else pd.DataFrame(columns=usecols)

print(f"CAM-ICU chartevents rows pulled (cancer-positive hadm_ids only): {len(cam_icu_events)}")
print()

# ============================================================
# STOP HERE — show the ACTUAL text values charted, per item, with counts
# No positive/negative pattern has been applied yet.
# ============================================================

if len(cam_icu_events) > 0:
    merged_labels = cam_icu_events.merge(
        cam_icu_items[['itemid', 'label']], on='itemid', how='left'
    )
    print("Actual charted values per CAM-ICU item (raw, unfiltered):")
    for itemid, label in cam_icu_items[['itemid', 'label']].values:
        subset = merged_labels[merged_labels['itemid'] == itemid]
        print(f"\n--- itemid {itemid} ({label}) — {len(subset)} rows ---")
        print(subset['value'].value_counts(dropna=False))
else:
    print("No CAM-ICU chartevents rows found for cancer-positive hadm_ids.")

In [ ]:
# ============================================================
# 4b. FLAG CAM-ICU POSITIVE per hadm_id
# Rule (per your spec): value starts with "Yes" (any variant) = feature present.
# "Unable to Assess" / "Language Barrier" / "Preexisting Advanced Dementia"
# are NOT counted as positive or negative — treated as missing/unknown.
# ============================================================

cam_icu_events['is_positive'] = cam_icu_events['value'].astype(str).str.strip().str.startswith('Yes')

cam_icu_positive_hadm_ids = set(
    cam_icu_events.loc[cam_icu_events['is_positive'], 'hadm_id']
)

print(f"CAM-ICU-positive hadm_id count: {len(cam_icu_positive_hadm_ids)}")
print()

# ============================================================
# 5. COMBINE: delirium-positive = ICD match OR CAM-ICU-positive
# ============================================================

delirium_positive_hadm_ids = delirium_icd_only_hadm_ids | cam_icu_positive_hadm_ids

print(f"Delirium-positive hadm_id count (ICD-only):        {len(delirium_icd_only_hadm_ids)}")
print(f"Delirium-positive hadm_id count (ICD + CAM-ICU):    {len(delirium_positive_hadm_ids)}")
print(f"  --> CAM-ICU contributed {len(delirium_positive_hadm_ids) - len(delirium_icd_only_hadm_ids)} additional hadm_ids")
print()

# ============================================================
# 6 & 7. CANCER-POSITIVE AND DELIRIUM-POSITIVE, KEEP BOTH TRUE
# ============================================================

cohort_hadm = hadm_flags[
    hadm_flags['hadm_id'].isin(cancer_positive_hadm_ids) &
    hadm_flags['hadm_id'].isin(delirium_positive_hadm_ids)
].copy()

print(f"Combined cancer+delirium hadm_id count: {len(cohort_hadm)}")
print(f"Unique subject_id at this stage: {cohort_hadm['subject_id'].nunique()}")
print()

# ============================================================
# 8. MERGE WITH icustays, FIRST ICU STAY ONLY, MERGE PATIENTS, AGE FILTER
# ============================================================

merged = cohort_hadm[['subject_id', 'hadm_id']].merge(
    icustays[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime']],
    on=['subject_id', 'hadm_id'],
    how='inner'
)

print(f"Rows after merging with icustays: {len(merged)}")

merged_sorted = merged.sort_values(['subject_id', 'intime'])
first_stay = merged_sorted.groupby('subject_id', as_index=False).first()

print(f"Count after first-ICU-stay-only filter: {len(first_stay)}")

first_stay = first_stay.merge(
    patients[['subject_id', 'anchor_age', 'anchor_year']],
    on='subject_id',
    how='left'
)

first_stay['age_at_admission'] = first_stay['anchor_age'] + (first_stay['intime'].dt.year - first_stay['anchor_year'])

cohort_step1 = first_stay[first_stay['age_at_admission'] >= 18].copy()

print(f"Count after age >= 18 filter: {len(cohort_step1)}")
print(f"Final unique subject_id count: {cohort_step1['subject_id'].nunique()}")
print()

# ============================================================
# 9. SUMMARY OF ALL COUNTS
# ============================================================

print("=" * 55)
print("SUMMARY")
print("=" * 55)
print(f"Cancer-positive hadm_id:                       {len(cancer_positive_hadm_ids)}")
print(f"Delirium-positive hadm_id (ICD-only):           {len(delirium_icd_only_hadm_ids)}")
print(f"Delirium-positive hadm_id (ICD + CAM-ICU):      {len(delirium_positive_hadm_ids)}")
print(f"Combined cancer+delirium hadm_id:               {len(cohort_hadm)}")
print(f"After first-ICU-stay-only:                      {len(first_stay)}")
print(f"After age >= 18 (FINAL):                        {len(cohort_step1)}")
print(f"Final unique subject_id:                        {cohort_step1['subject_id'].nunique()}")

# ============================================================
# 10. OVERWRITE cohort_step1.csv
# ============================================================

output_cols = ['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'age_at_admission']
cohort_step1[output_cols].to_csv('../data/processed/cohort_step1.csv', index=False)

print()
print(f"Saved cohort_step1.csv with {len(cohort_step1)} rows to ../data/processed/cohort_step1.csv")

In [ ]:
# ============================================================
# 4c. PROPER 4-FEATURE CAM-ICU RULE (replaces the "any single Yes" rule)
# Feature 1 = MS Change, Feature 2 = Inattention,
# Feature 3 = Disorganized Thinking, Feature 4 = Altered LOC
# RASS LOC (228302) excluded — it's a gating/assessability item, not a diagnostic feature.
# Diagnosis = F1 AND F2 AND (F3 OR F4), evaluated per (hadm_id, charttime) assessment.
# ============================================================

feature_map = {
    228300: 'ms_change', 228337: 'ms_change', 229326: 'ms_change',
    228301: 'inattention', 228336: 'inattention', 229325: 'inattention',
    228303: 'disorganized_thinking', 228335: 'disorganized_thinking', 229324: 'disorganized_thinking',
    228334: 'altered_loc',
    228302: 'rass_loc'  # excluded from diagnostic combination below
}

cam_icu_events['feature'] = cam_icu_events['itemid'].map(feature_map)
cam_icu_events['is_yes'] = cam_icu_events['value'].astype(str).str.strip().str.startswith('Yes')

# Drop the gating item — not part of the diagnostic rule
diagnostic_events = cam_icu_events[cam_icu_events['feature'] != 'rass_loc'].copy()

# One row per assessment (hadm_id + charttime), one column per feature
assessment_pivot = diagnostic_events.pivot_table(
    index=['hadm_id', 'charttime'],
    columns='feature',
    values='is_yes',
    aggfunc='max'  # True if any row for that feature/time was "Yes"
).reset_index()

for col in ['ms_change', 'inattention', 'disorganized_thinking', 'altered_loc']:
    if col not in assessment_pivot.columns:
        assessment_pivot[col] = False
assessment_pivot = assessment_pivot.fillna(False)

assessment_pivot['cam_icu_positive'] = (
    assessment_pivot['ms_change'] &
    assessment_pivot['inattention'] &
    (assessment_pivot['disorganized_thinking'] | assessment_pivot['altered_loc'])
)

print(f"Total CAM-ICU assessments reconstructed: {len(assessment_pivot)}")
print(f"Assessments meeting full 4-feature positive rule: {assessment_pivot['cam_icu_positive'].sum()}")
print()

cam_icu_positive_hadm_ids = set(
    assessment_pivot.loc[assessment_pivot['cam_icu_positive'], 'hadm_id']
)

print(f"CAM-ICU-positive hadm_id count (proper 4-feature rule): {len(cam_icu_positive_hadm_ids)}")
print(f"  (previous loose 'any single Yes' rule gave: 4449 — for comparison)")
print()

# ============================================================
# 5. COMBINE: delirium-positive = ICD match OR CAM-ICU-positive
# ============================================================

delirium_positive_hadm_ids = delirium_icd_only_hadm_ids | cam_icu_positive_hadm_ids

print(f"Delirium-positive hadm_id count (ICD-only):        {len(delirium_icd_only_hadm_ids)}")
print(f"Delirium-positive hadm_id count (ICD + CAM-ICU):    {len(delirium_positive_hadm_ids)}")
print(f"  --> CAM-ICU contributed {len(delirium_positive_hadm_ids) - len(delirium_icd_only_hadm_ids)} additional hadm_ids")
print()

# ============================================================
# 6 & 7. CANCER-POSITIVE AND DELIRIUM-POSITIVE, KEEP BOTH TRUE
# ============================================================

cohort_hadm = hadm_flags[
    hadm_flags['hadm_id'].isin(cancer_positive_hadm_ids) &
    hadm_flags['hadm_id'].isin(delirium_positive_hadm_ids)
].copy()

print(f"Combined cancer+delirium hadm_id count: {len(cohort_hadm)}")
print(f"Unique subject_id at this stage: {cohort_hadm['subject_id'].nunique()}")
print()

# ============================================================
# 8. MERGE WITH icustays, FIRST ICU STAY ONLY, MERGE PATIENTS, AGE FILTER
# ============================================================

merged = cohort_hadm[['subject_id', 'hadm_id']].merge(
    icustays[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime']],
    on=['subject_id', 'hadm_id'],
    how='inner'
)

print(f"Rows after merging with icustays: {len(merged)}")

merged_sorted = merged.sort_values(['subject_id', 'intime'])
first_stay = merged_sorted.groupby('subject_id', as_index=False).first()

print(f"Count after first-ICU-stay-only filter: {len(first_stay)}")

first_stay = first_stay.merge(
    patients[['subject_id', 'anchor_age', 'anchor_year']],
    on='subject_id',
    how='left'
)

first_stay['age_at_admission'] = first_stay['anchor_age'] + (first_stay['intime'].dt.year - first_stay['anchor_year'])

cohort_step1 = first_stay[first_stay['age_at_admission'] >= 18].copy()

print(f"Count after age >= 18 filter: {len(cohort_step1)}")
print(f"Final unique subject_id count: {cohort_step1['subject_id'].nunique()}")
print()

# ============================================================
# 9. SUMMARY
# ============================================================

print("=" * 55)
print("SUMMARY (proper 4-feature CAM-ICU rule)")
print("=" * 55)
print(f"Cancer-positive hadm_id:                       {len(cancer_positive_hadm_ids)}")
print(f"Delirium-positive hadm_id (ICD-only):           {len(delirium_icd_only_hadm_ids)}")
print(f"Delirium-positive hadm_id (ICD + CAM-ICU):      {len(delirium_positive_hadm_ids)}")
print(f"Combined cancer+delirium hadm_id:               {len(cohort_hadm)}")
print(f"After first-ICU-stay-only:                      {len(first_stay)}")
print(f"After age >= 18 (FINAL):                        {len(cohort_step1)}")
print(f"Final unique subject_id:                        {cohort_step1['subject_id'].nunique()}")

# ============================================================
# 10. OVERWRITE cohort_step1.csv
# ============================================================

output_cols = ['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'age_at_admission']
cohort_step1[output_cols].to_csv('../data/processed/cohort_step1.csv', index=False)

print()
print(f"Saved cohort_step1.csv with {len(cohort_step1)} rows to ../data/processed/cohort_step1.csv")

In [ ]:
import pandas as pd

# Reload the current cohort_step1.csv (the saved state) and icustays raw,
# to be self-contained regardless of what's still in session memory.

raw_path = "../data/raw/"
processed_path = "../data/processed/"

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
cohort_step1['outtime'] = pd.to_datetime(cohort_step1['outtime'])

icustays_raw = pd.read_csv(raw_path + "icustays.csv.gz", compression="gzip")
icustays_raw['intime'] = pd.to_datetime(icustays_raw['intime'])
icustays_raw['outtime'] = pd.to_datetime(icustays_raw['outtime'])

# ============================================================
# 1. Find the row(s) with missing outtime in cohort_step1
# ============================================================

missing_outtime = cohort_step1[cohort_step1['outtime'].isna()]

print(f"Rows with missing outtime in cohort_step1: {len(missing_outtime)}")
print(missing_outtime.to_string(index=False))
print()

if len(missing_outtime) == 0:
    print("No missing outtime found — nothing to fix.")
else:
    for _, row in missing_outtime.iterrows():
        subject_id = row['subject_id']
        hadm_id = row['hadm_id']
        stay_id = row['stay_id']

        print(f"Investigating subject_id={subject_id}, hadm_id={hadm_id}, stay_id={stay_id}")

        # ============================================================
        # 2. Check the original icustays table for this stay_id
        # ============================================================

        source_row = icustays_raw[icustays_raw['stay_id'] == stay_id]

        print("Matching row(s) in original icustays table:")
        print(source_row.to_string(index=False))
        print()

        if len(source_row) == 0:
            print(f"stay_id {stay_id} not found in icustays at all — flagging for exclusion.")
            cohort_step1 = cohort_step1[cohort_step1['stay_id'] != stay_id]
            continue

        src_outtime = source_row['outtime'].iloc[0]
        src_los = source_row['los'].iloc[0]
        src_intime = source_row['intime'].iloc[0]

        if pd.notna(src_outtime):
            print(f"outtime IS present in source icustays ({src_outtime}) — was lost in an earlier merge. Restoring it.")
            cohort_step1.loc[cohort_step1['stay_id'] == stay_id, 'outtime'] = src_outtime

        elif pd.notna(src_los):
            computed_outtime = src_intime + pd.Timedelta(days=src_los)
            print(f"outtime missing in source, but los={src_los} days is present.")
            print(f"Computing outtime = intime + los = {src_intime} + {src_los} days = {computed_outtime}")
            cohort_step1.loc[cohort_step1['stay_id'] == stay_id, 'outtime'] = computed_outtime

        else:
            print(f"BOTH outtime AND los are missing in source icustays for stay_id={stay_id}.")
            print(f"Cannot recover discharge time for this patient — flagging subject_id={subject_id}, "
                  f"hadm_id={hadm_id}, stay_id={stay_id} for EXCLUSION.")
            cohort_step1 = cohort_step1[cohort_step1['stay_id'] != stay_id]

        print()

# ============================================================
# 3. Re-save cohort_step1.csv
# ============================================================

output_cols = ['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'age_at_admission']
cohort_step1[output_cols].to_csv(processed_path + "cohort_step1.csv", index=False)

print(f"Re-saved cohort_step1.csv with {len(cohort_step1)} rows.")
print(f"Remaining rows with missing outtime: {cohort_step1['outtime'].isna().sum()}")

In [ ]:
import pandas as pd

raw_path = "../data/raw/"
processed_path = "../data/processed/"

# ============================================================
# Reload cohort_step1.csv, admissions, patients fresh
# ============================================================

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
cohort_step1['outtime'] = pd.to_datetime(cohort_step1['outtime'])

admissions = pd.read_csv(raw_path + "admissions.csv.gz", compression="gzip")
admissions['deathtime'] = pd.to_datetime(admissions['deathtime'])
admissions['admittime'] = pd.to_datetime(admissions['admittime'])
admissions['dischtime'] = pd.to_datetime(admissions['dischtime'])

patients = pd.read_csv(raw_path + "patients.csv.gz", compression="gzip")

print(f"cohort_step1 loaded: {cohort_step1.shape}")
print(f"admissions loaded: {admissions.shape}")
print(f"patients loaded: {patients.shape}")
print(f"patients columns (checking for dod): {patients.columns.tolist()}")
print()

if 'dod' not in patients.columns:
    raise ValueError("patients table does not contain a 'dod' column — cannot proceed as specified.")

patients['dod'] = pd.to_datetime(patients['dod'])

# ============================================================
# 1. Merge in deathtime (via hadm_id) and dod (via subject_id)
# ============================================================

cohort = cohort_step1.merge(
    admissions[['subject_id', 'hadm_id', 'deathtime']],
    on=['subject_id', 'hadm_id'],
    how='left'
)

cohort = cohort.merge(
    patients[['subject_id', 'dod']],
    on='subject_id',
    how='left'
)

# Determine which death timestamp to use: deathtime first, else dod
cohort['death_source'] = None
cohort.loc[cohort['deathtime'].notna(), 'death_source'] = 'deathtime'
cohort.loc[cohort['deathtime'].isna() & cohort['dod'].notna(), 'death_source'] = 'dod'

cohort['death_timestamp'] = cohort['deathtime']
cohort.loc[cohort['death_source'] == 'dod', 'death_timestamp'] = cohort.loc[cohort['death_source'] == 'dod', 'dod']

n_via_deathtime = (cohort['death_source'] == 'deathtime').sum()
n_via_dod = (cohort['death_source'] == 'dod').sum()
n_no_death = cohort['death_source'].isna().sum()

print(f"Mortality determined via deathtime: {n_via_deathtime}")
print(f"Mortality determined via dod (fallback): {n_via_dod}")
print(f"No death record found (alive / no data): {n_no_death}")
print()

# ============================================================
# 2. 28-day mortality label
# ============================================================

cohort['days_to_death'] = (cohort['death_timestamp'] - cohort['intime']).dt.total_seconds() / 86400

cohort['mortality_28d'] = (
    cohort['death_timestamp'].notna() &
    (cohort['days_to_death'] >= 0) &
    (cohort['days_to_death'] <= 28)
)

# ============================================================
# 3. Flag (not drop) immortal-time cases: death within 24 hours of ICU intime
# ============================================================

cohort['immortal_time_flag'] = (
    cohort['death_timestamp'].notna() &
    (cohort['days_to_death'] >= 0) &
    (cohort['days_to_death'] < 1)
)

# ============================================================
# 4. Print summary stats
# ============================================================

mortality_count = cohort['mortality_28d'].sum()
mortality_rate = 100 * mortality_count / len(cohort)
immortal_count = cohort['immortal_time_flag'].sum()

print("=" * 55)
print("SUMMARY")
print("=" * 55)
print(f"Total patients: {len(cohort)}")
print(f"28-day mortality count: {mortality_count}")
print(f"28-day mortality rate: {mortality_rate:.2f}%")
print(f"immortal_time_flag count (died within 24h of ICU intime): {immortal_count}")
print()

# Sanity check: any negative days_to_death (death before ICU intime)? Shouldn't happen but worth flagging.
negative_days = (cohort['days_to_death'] < 0).sum()
if negative_days > 0:
    print(f"WARNING: {negative_days} rows have a death_timestamp BEFORE icu intime — check these manually.")
    print(cohort[cohort['days_to_death'] < 0][['subject_id', 'hadm_id', 'stay_id', 'intime', 'death_timestamp', 'death_source']].to_string(index=False))
    print()

# ============================================================
# 5. Save updated cohort_step1.csv
# ============================================================

output_cols = [
    'subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime',
    'age_at_admission', 'mortality_28d', 'immortal_time_flag'
]

cohort_final = cohort[output_cols].copy()
cohort_final.to_csv(processed_path + "cohort_step1.csv", index=False)

print(f"Saved cohort_step1.csv with mortality_28d and immortal_time_flag columns.")
print(f"Final shape: {cohort_final.shape}")

In [ ]:
import pandas as pd
import re

raw_path = "../data/raw/"
processed_path = "../data/processed/"

# ============================================================
# Reload fresh
# ============================================================

d_icd_diagnoses = pd.read_csv(raw_path + "d_icd_diagnoses.csv.gz", compression="gzip")
d_icd_diagnoses['icd_code'] = d_icd_diagnoses['icd_code'].astype(str).str.strip()

diagnoses_icd = pd.read_csv(raw_path + "diagnoses_icd.csv.gz", compression="gzip")
diagnoses_icd['icd_code'] = diagnoses_icd['icd_code'].astype(str).str.strip()

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
print(f"cohort_step1.csv loaded: {cohort_step1.shape}")

cohort_hadm_ids = set(cohort_step1['hadm_id'])
print(f"Unique hadm_id in cohort_step1.csv: {len(cohort_hadm_ids)}")
print()

# ============================================================
# Shared cancer rule logic
# ============================================================

def icd9_category(code):
    code = str(code)
    return int(code[:3]) if code[:3].isdigit() else None

def icd10_cancer_match(code):
    m = re.match(r'^C(\d{2})', str(code))
    return 0 <= int(m.group(1)) <= 96 if m else False

def cancer_range_match(row):
    code = str(row['icd_code'])
    if row['icd_version'] == 9:
        cat = icd9_category(code)
        if cat is None:
            return False
        return (140 <= cat <= 172) or (174 <= cat <= 189) or (190 <= cat <= 209)
    elif row['icd_version'] == 10:
        return icd10_cancer_match(code)
    return False

def is_carcinoma_in_situ(row):
    code = str(row['icd_code'])
    if row['icd_version'] == 9:
        cat = icd9_category(code)
        return cat is not None and 230 <= cat <= 234
    elif row['icd_version'] == 10:
        return code[:2] == 'D0'
    return False

exclude_terms = [
    'in remission', 'personal history', 'family history', 'screening',
    'genetic susceptibility', 'follow-up', 'prophylactic',
    'complicating pregnancy', 'complicating childbirth', 'complicating the puerperium'
]
exclude_pattern = '|'.join(exclude_terms)
explicit_exclude_codes = {'79582', 'R971', 'R9721', 'D701'}

# ============================================================
# 1 & 2. VERSION A: full-table scan (the 2052-code version)
# ============================================================

range_mask_full = d_icd_diagnoses.apply(cancer_range_match, axis=1)
term_mask_full = d_icd_diagnoses['long_title'].str.contains(exclude_pattern, case=False, na=False, regex=True)
explicit_mask_full = d_icd_diagnoses['icd_code'].isin(explicit_exclude_codes)
cis_mask_full = d_icd_diagnoses.apply(is_carcinoma_in_situ, axis=1)

cancer_include_full = range_mask_full & ~term_mask_full & ~explicit_mask_full & ~cis_mask_full
cancer_codes_full = set(d_icd_diagnoses.loc[cancer_include_full, 'icd_code'])

print(f"VERSION A (full-table scan): {len(cancer_codes_full)} cancer codes kept")

diagnoses_icd['is_cancer_full'] = diagnoses_icd['icd_code'].isin(cancer_codes_full)
cancer_positive_hadm_full = set(
    diagnoses_icd.loc[diagnoses_icd['is_cancer_full'], 'hadm_id']
)
print(f"Cancer-positive hadm_id under VERSION A: {len(cancer_positive_hadm_full)}")
print()

# ============================================================
# 5. VERSION B: keyword-prefiltered subset (the 1706-code version)
# This replicates the EARLIER approach: first keyword-search long_title,
# THEN apply the same range/exclusion rule only within that subset.
# ============================================================

cancer_keyword_terms = ['malignant neoplasm', 'cancer', 'carcinoma', 'leukemia', 'lymphoma']
cancer_keyword_pattern = '|'.join(cancer_keyword_terms)

cancer_candidates_subset = d_icd_diagnoses[
    d_icd_diagnoses['long_title'].str.contains(cancer_keyword_pattern, case=False, na=False, regex=True)
].copy()

range_mask_sub = cancer_candidates_subset.apply(cancer_range_match, axis=1)
term_mask_sub = cancer_candidates_subset['long_title'].str.contains(exclude_pattern, case=False, na=False, regex=True)
explicit_mask_sub = cancer_candidates_subset['icd_code'].isin(explicit_exclude_codes)
cis_mask_sub = cancer_candidates_subset.apply(is_carcinoma_in_situ, axis=1)

cancer_include_sub = range_mask_sub & ~term_mask_sub & ~explicit_mask_sub & ~cis_mask_sub
cancer_codes_subset = set(cancer_candidates_subset.loc[cancer_include_sub, 'icd_code'])

print(f"VERSION B (keyword-prefiltered subset): {len(cancer_codes_subset)} cancer codes kept")

diagnoses_icd['is_cancer_sub'] = diagnoses_icd['icd_code'].isin(cancer_codes_subset)
cancer_positive_hadm_sub = set(
    diagnoses_icd.loc[diagnoses_icd['is_cancer_sub'], 'hadm_id']
)
print(f"Cancer-positive hadm_id under VERSION B: {len(cancer_positive_hadm_sub)}")
print()

# ============================================================
# 4. Check cohort_step1 hadm_id against VERSION A
# ============================================================

match_A = cohort_hadm_ids & cancer_positive_hadm_full
mismatch_A = cohort_hadm_ids - cancer_positive_hadm_full

print("=" * 55)
print(f"CHECK vs VERSION A ({len(cancer_codes_full)} codes):")
print(f"  hadm_id in cohort_step1 that ARE cancer-positive: {len(match_A)}")
print(f"  hadm_id in cohort_step1 that are NOT cancer-positive: {len(mismatch_A)}")
print(f"  Match rate: {100*len(match_A)/len(cohort_hadm_ids):.2f}%")
print()

# ============================================================
# Check cohort_step1 hadm_id against VERSION B
# ============================================================

match_B = cohort_hadm_ids & cancer_positive_hadm_sub
mismatch_B = cohort_hadm_ids - cancer_positive_hadm_sub

print("=" * 55)
print(f"CHECK vs VERSION B ({len(cancer_codes_subset)} codes):")
print(f"  hadm_id in cohort_step1 that ARE cancer-positive: {len(match_B)}")
print(f"  hadm_id in cohort_step1 that are NOT cancer-positive: {len(mismatch_B)}")
print(f"  Match rate: {100*len(match_B)/len(cohort_hadm_ids):.2f}%")
print()

# ============================================================
# 6. VERDICT
# ============================================================

print("=" * 55)
print("VERDICT")
print("=" * 55)

a_full_match = (len(mismatch_A) == 0)
b_full_match = (len(mismatch_B) == 0)

if a_full_match and not b_full_match:
    print(f"VERSION A ({len(cancer_codes_full)} codes) is the list that built cohort_step1.csv.")
    print(f"100% of cohort hadm_id are cancer-positive under VERSION A; VERSION B leaves {len(mismatch_B)} unexplained.")
elif b_full_match and not a_full_match:
    print(f"VERSION B ({len(cancer_codes_subset)} codes) is the list that built cohort_step1.csv.")
    print(f"100% of cohort hadm_id are cancer-positive under VERSION B; VERSION A leaves {len(mismatch_A)} unexplained.")
elif a_full_match and b_full_match:
    print("Both lists are consistent with 100% of cohort_step1.csv — cannot distinguish from this check alone.")
else:
    print("NEITHER list fully explains cohort_step1.csv's hadm_id.")
    print(f"  VERSION A leaves {len(mismatch_A)} hadm_id unexplained.")
    print(f"  VERSION B leaves {len(mismatch_B)} hadm_id unexplained.")
    print("  This suggests cohort_step1.csv was built from a DIFFERENT cancer code list than either")
    print("  reconstruction here, OR the delirium/first-ICU-stay/age filters removed hadm_id in a way")
    print("  that breaks this check. Worth re-tracing the exact cell sequence that produced the saved file.")

In [ ]:
import pandas as pd
import re

raw_path = "../data/raw/"
processed_path = "../data/processed/"

# ============================================================
# Reload everything fresh
# ============================================================

d_icd_diagnoses = pd.read_csv(raw_path + "d_icd_diagnoses.csv.gz", compression="gzip")
d_icd_diagnoses['icd_code'] = d_icd_diagnoses['icd_code'].astype(str).str.strip()

diagnoses_icd = pd.read_csv(raw_path + "diagnoses_icd.csv.gz", compression="gzip")
diagnoses_icd['icd_code'] = diagnoses_icd['icd_code'].astype(str).str.strip()

d_items = pd.read_csv(raw_path + "d_items.csv.gz", compression="gzip")

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_hadm_ids = set(cohort_step1['hadm_id'])
print(f"cohort_step1.csv loaded: {cohort_step1.shape}")
print(f"Unique hadm_id in cohort_step1.csv: {len(cohort_hadm_ids)}")
print()

# ============================================================
# 1. Rebuild the CONFIRMED 2052-code cancer list (full-table scan)
# ============================================================

def icd9_category(code):
    code = str(code)
    return int(code[:3]) if code[:3].isdigit() else None

def icd10_cancer_match(code):
    m = re.match(r'^C(\d{2})', str(code))
    return 0 <= int(m.group(1)) <= 96 if m else False

def cancer_range_match(row):
    code = str(row['icd_code'])
    if row['icd_version'] == 9:
        cat = icd9_category(code)
        if cat is None:
            return False
        return (140 <= cat <= 172) or (174 <= cat <= 189) or (190 <= cat <= 209)
    elif row['icd_version'] == 10:
        return icd10_cancer_match(code)
    return False

def is_carcinoma_in_situ(row):
    code = str(row['icd_code'])
    if row['icd_version'] == 9:
        cat = icd9_category(code)
        return cat is not None and 230 <= cat <= 234
    elif row['icd_version'] == 10:
        return code[:2] == 'D0'
    return False

exclude_terms = [
    'in remission', 'personal history', 'family history', 'screening',
    'genetic susceptibility', 'follow-up', 'prophylactic',
    'complicating pregnancy', 'complicating childbirth', 'complicating the puerperium'
]
exclude_pattern = '|'.join(exclude_terms)
explicit_exclude_codes = {'79582', 'R971', 'R9721', 'D701'}

range_mask = d_icd_diagnoses.apply(cancer_range_match, axis=1)
term_mask = d_icd_diagnoses['long_title'].str.contains(exclude_pattern, case=False, na=False, regex=True)
explicit_mask = d_icd_diagnoses['icd_code'].isin(explicit_exclude_codes)
cis_mask = d_icd_diagnoses.apply(is_carcinoma_in_situ, axis=1)

cancer_include_mask = range_mask & ~term_mask & ~explicit_mask & ~cis_mask
cancer_final_codes = set(d_icd_diagnoses.loc[cancer_include_mask, 'icd_code'])

print(f"Cancer codes kept: {len(cancer_final_codes)} (expected 2052)")

diagnoses_icd['is_cancer'] = diagnoses_icd['icd_code'].isin(cancer_final_codes)
hadm_flags = diagnoses_icd.groupby('hadm_id').agg(
    subject_id=('subject_id', 'first'),
    cancer=('is_cancer', 'any')
).reset_index()

cancer_positive_hadm_ids = set(hadm_flags.loc[hadm_flags['cancer'], 'hadm_id'])
print(f"Cancer-positive hadm_id: {len(cancer_positive_hadm_ids)} (expected 74503)")
print()

# ============================================================
# 2. Rebuild delirium-positive hadm_id: ICD (all patients) + CAM-ICU
#    (CAM-ICU chartevents pulled ONLY for cancer-positive hadm_id, matching original pipeline)
# ============================================================

delirium_final_codes = {'F05', '2930', '2931'}
diagnoses_icd['is_delirium_icd'] = diagnoses_icd['icd_code'].isin(delirium_final_codes)

delirium_icd_flags = diagnoses_icd.groupby('hadm_id')['is_delirium_icd'].any().reset_index()
delirium_icd_only_hadm_ids = set(delirium_icd_flags.loc[delirium_icd_flags['is_delirium_icd'], 'hadm_id'])
print(f"Delirium-positive hadm_id (ICD-only, all patients): {len(delirium_icd_only_hadm_ids)}")

cam_icu_items = d_items[
    d_items['label'].str.contains('CAM-ICU', case=False, na=False) |
    d_items['label'].str.contains('confusion assessment', case=False, na=False)
].copy()
cam_icu_itemids = set(cam_icu_items['itemid'])

feature_map = {
    228300: 'ms_change', 228337: 'ms_change', 229326: 'ms_change',
    228301: 'inattention', 228336: 'inattention', 229325: 'inattention',
    228303: 'disorganized_thinking', 228335: 'disorganized_thinking', 229324: 'disorganized_thinking',
    228334: 'altered_loc',
    228302: 'rass_loc'
}

chunk_size = 1_000_000
cam_icu_rows = []
usecols = ['subject_id', 'hadm_id', 'itemid', 'charttime', 'value', 'valuenum']

for chunk in pd.read_csv(
    raw_path + "chartevents.csv.gz",
    compression="gzip",
    usecols=usecols,
    chunksize=chunk_size
):
    filtered = chunk[
        chunk['itemid'].isin(cam_icu_itemids) &
        chunk['hadm_id'].isin(cancer_positive_hadm_ids)  # NOTE: only cancer-positive hadm_id pulled, matching original
    ]
    if len(filtered) > 0:
        cam_icu_rows.append(filtered)

cam_icu_events = pd.concat(cam_icu_rows, ignore_index=True) if cam_icu_rows else pd.DataFrame(columns=usecols)
print(f"CAM-ICU chartevents rows pulled (cancer-positive hadm_id only): {len(cam_icu_events)}")

cam_icu_events['feature'] = cam_icu_events['itemid'].map(feature_map)
cam_icu_events['is_yes'] = cam_icu_events['value'].astype(str).str.strip().str.startswith('Yes')

diagnostic_events = cam_icu_events[cam_icu_events['feature'] != 'rass_loc'].copy()

assessment_pivot = diagnostic_events.pivot_table(
    index=['hadm_id', 'charttime'],
    columns='feature',
    values='is_yes',
    aggfunc='max'
).reset_index()

for col in ['ms_change', 'inattention', 'disorganized_thinking', 'altered_loc']:
    if col not in assessment_pivot.columns:
        assessment_pivot[col] = False
assessment_pivot = assessment_pivot.fillna(False)

assessment_pivot['cam_icu_positive'] = (
    assessment_pivot['ms_change'] &
    assessment_pivot['inattention'] &
    (assessment_pivot['disorganized_thinking'] | assessment_pivot['altered_loc'])
)

cam_icu_positive_hadm_ids = set(
    assessment_pivot.loc[assessment_pivot['cam_icu_positive'], 'hadm_id']
)
print(f"CAM-ICU-positive hadm_id (4-feature rule, within cancer-positive only): {len(cam_icu_positive_hadm_ids)}")

delirium_positive_hadm_ids = delirium_icd_only_hadm_ids | cam_icu_positive_hadm_ids
print(f"Delirium-positive hadm_id (ICD[all] UNION CAM-ICU[cancer-positive only]): {len(delirium_positive_hadm_ids)}")
print("  NOTE: this is NOT a universal delirium-positive count — CAM-ICU component was only")
print("  ever evaluated within cancer-positive patients, per the original pipeline's chartevents filter.")
print()

# ============================================================
# 3. Intersection: cancer-positive AND delirium-positive
# ============================================================

combined_hadm_ids = cancer_positive_hadm_ids & delirium_positive_hadm_ids
print(f"Combined cancer+delirium hadm_id: {len(combined_hadm_ids)} (expected ~4361)")

# ============================================================
# 4. Unique subject_id from that intersection
# ============================================================

combined_subject_ids = set(
    hadm_flags.loc[hadm_flags['hadm_id'].isin(combined_hadm_ids), 'subject_id']
)
print(f"Unique subject_id from combined intersection: {len(combined_subject_ids)} (expected ~3950)")
print()

# ============================================================
# 5. Check: every hadm_id in cohort_step1.csv also in combined_hadm_ids?
# ============================================================

match_combined = cohort_hadm_ids & combined_hadm_ids
mismatch_combined = cohort_hadm_ids - combined_hadm_ids

print("=" * 55)
print("CHECK: cohort_step1.csv hadm_id vs rebuilt cancer+delirium intersection")
print(f"  Match: {len(match_combined)}")
print(f"  Mismatch: {len(mismatch_combined)}")
print()

# ============================================================
# 6. If mismatch: isolate whether it's cancer side, delirium side, or intersection logic
# ============================================================

if len(mismatch_combined) > 0:
    print("Isolating mismatch source for each unmatched hadm_id:")
    diag_rows = []
    for hid in mismatch_combined:
        in_cancer = hid in cancer_positive_hadm_ids
        in_delirium = hid in delirium_positive_hadm_ids
        diag_rows.append({'hadm_id': hid, 'cancer_positive': in_cancer, 'delirium_positive': in_delirium})
    diag_df = pd.DataFrame(diag_rows)
    print(diag_df.to_string(index=False))
    print()
    print(f"  Mismatches missing cancer flag only: {((~diag_df['cancer_positive']) & diag_df['delirium_positive']).sum()}")
    print(f"  Mismatches missing delirium flag only: {(diag_df['cancer_positive'] & (~diag_df['delirium_positive'])).sum()}")
    print(f"  Mismatches missing BOTH: {((~diag_df['cancer_positive']) & (~diag_df['delirium_positive'])).sum()}")
    print()

# ============================================================
# 7. FINAL VERDICT
# ============================================================

print("=" * 55)
print("FINAL VERDICT")
print("=" * 55)

cancer_ok = (len(cancer_positive_hadm_ids) == 74503)
combined_matches_all = (len(mismatch_combined) == 0)

if cancer_ok and combined_matches_all:
    print("ALL STAGE NUMBERS ARE INTERNALLY CONSISTENT.")
    print(f"  Cancer-positive: {len(cancer_positive_hadm_ids)} (matches expected 74503)")
    print(f"  Combined cancer+delirium: {len(combined_hadm_ids)}")
    print(f"  100% of cohort_step1.csv hadm_id are explained by this rebuild.")
    print("  Safe to use these numbers in the attrition diagram.")
elif not cancer_ok:
    print(f"MISMATCH: cancer-positive count is {len(cancer_positive_hadm_ids)}, not the expected 74503.")
    print("  The cancer code list or diagnoses_icd may have changed between runs — re-verify before proceeding.")
else:
    print(f"MISMATCH: {len(mismatch_combined)} hadm_id in cohort_step1.csv are NOT explained by this")
    print("  freshly-rebuilt cancer+delirium intersection. See the isolation table above for whether")
    print("  the gap is coming from the cancer flag, delirium flag, or both.")
    print("  Do NOT use the 12309/4361/3950 numbers in the attrition diagram until this is resolved —")
    print("  re-trace the exact cell sequence that produced the saved cohort_step1.csv instead.")

In [ ]:
import pandas as pd
import re

raw_path = "../data/raw/"
processed_path = "../data/processed/"

d_icd_diagnoses = pd.read_csv(raw_path + "d_icd_diagnoses.csv.gz", compression="gzip")
d_icd_diagnoses['icd_code'] = d_icd_diagnoses['icd_code'].astype(str).str.strip()

# ============================================================
# Rebuild the confirmed cancer rule (full-table-scan version, 2052 codes)
# ============================================================

def icd9_category(code):
    code = str(code)
    return int(code[:3]) if code[:3].isdigit() else None

def icd10_cancer_match(code):
    m = re.match(r'^C(\d{2})', str(code))
    return 0 <= int(m.group(1)) <= 96 if m else False

def cancer_range_match(row):
    code = str(row['icd_code'])
    if row['icd_version'] == 9:
        cat = icd9_category(code)
        if cat is None:
            return False
        return (140 <= cat <= 172) or (174 <= cat <= 189) or (190 <= cat <= 209)
    elif row['icd_version'] == 10:
        return icd10_cancer_match(code)
    return False

def is_carcinoma_in_situ(row):
    code = str(row['icd_code'])
    if row['icd_version'] == 9:
        cat = icd9_category(code)
        return cat is not None and 230 <= cat <= 234
    elif row['icd_version'] == 10:
        return code[:2] == 'D0'
    return False

exclude_terms = [
    'in remission', 'personal history', 'family history', 'screening',
    'genetic susceptibility', 'follow-up', 'prophylactic',
    'complicating pregnancy', 'complicating childbirth', 'complicating the puerperium'
]
explicit_exclude_codes = {'79582', 'R971', 'R9721', 'D701'}

def matched_term(title):
    title_lower = str(title).lower()
    for term in exclude_terms:
        if term in title_lower:
            return term
    return None

d_icd_diagnoses['matched_exclude_term'] = d_icd_diagnoses['long_title'].apply(matched_term)

range_mask = d_icd_diagnoses.apply(cancer_range_match, axis=1)
term_mask = d_icd_diagnoses['matched_exclude_term'].notna()
explicit_mask = d_icd_diagnoses['icd_code'].isin(explicit_exclude_codes)
cis_mask = d_icd_diagnoses.apply(is_carcinoma_in_situ, axis=1)

cancer_include_mask = range_mask & ~term_mask & ~explicit_mask & ~cis_mask
cancer_excluded_mask = range_mask & ~cancer_include_mask  # candidates by range, but dropped by a rule

total_cancer_candidates = range_mask.sum()
n_kept = cancer_include_mask.sum()
n_excluded = cancer_excluded_mask.sum()

print(f"Total cancer range candidates (before exclusion rules): {total_cancer_candidates}")
print(f"Kept: {n_kept} (expected 2052)")
print(f"Excluded: {n_excluded}")
print(f"Kept + Excluded should equal total candidates: {n_kept + n_excluded} == {total_cancer_candidates} -> {n_kept + n_excluded == total_cancer_candidates}")
print()

# ============================================================
# 1. Appendix table: kept cancer codes + delirium codes
# ============================================================

cancer_kept = d_icd_diagnoses.loc[cancer_include_mask, ['icd_code', 'icd_version', 'long_title']].copy()
cancer_kept['category'] = 'cancer'

delirium_codes_lookup = {'F05': 10, '2930': 9, '2931': 9}
delirium_rows = d_icd_diagnoses[
    d_icd_diagnoses.apply(lambda r: r['icd_code'] in delirium_codes_lookup
                           and r['icd_version'] == delirium_codes_lookup[r['icd_code']], axis=1)
][['icd_code', 'icd_version', 'long_title']].copy()
delirium_rows['category'] = 'delirium'

print(f"Cancer rows for appendix: {len(cancer_kept)}")
print(f"Delirium ICD rows for appendix: {len(delirium_rows)}")

cam_icu_note = pd.DataFrame([{
    'icd_code': 'N/A',
    'icd_version': 'N/A',
    'long_title': ("Delirium ascertainment: ICD codes F05, 293.0, 293.1 checked across ALL patients, "
                   "PLUS CAM-ICU 4-feature screening (acute/fluctuating mental status change AND "
                   "inattention AND [disorganized thinking OR altered level of consciousness]) "
                   "checked among cancer-positive patients only. This row documents the CAM-ICU "
                   "component, which is not itself an ICD code."),
    'category': 'delirium (CAM-ICU, non-ICD)'
}])

appendix_table = pd.concat([cancer_kept, delirium_rows, cam_icu_note], ignore_index=True)
appendix_table = appendix_table.sort_values(['category', 'icd_version', 'icd_code'], na_position='last').reset_index(drop=True)

appendix_table.to_csv(processed_path + "icd_code_appendix.csv", index=False)

# ============================================================
# 2. Exclusion table with reason per code
# ============================================================

def build_reason(row):
    code = row['icd_code']
    if code in explicit_exclude_codes:
        return "explicitly excluded code (lab marker / follow-up code, not a diagnosis of malignancy)"
    if is_carcinoma_in_situ(row):
        return "carcinoma in situ (non-invasive; excluded from invasive malignancy definition)"
    if row['matched_exclude_term'] is not None:
        return f"long_title contains excluded term: '{row['matched_exclude_term']}'"
    return "unmatched — check logic"  # should not occur

excluded_df = d_icd_diagnoses.loc[cancer_excluded_mask, ['icd_code', 'long_title']].copy()
excluded_df['exclusion_reason'] = d_icd_diagnoses.loc[cancer_excluded_mask].apply(build_reason, axis=1)
excluded_df = excluded_df.sort_values('icd_code').reset_index(drop=True)

excluded_df.to_csv(processed_path + "icd_code_exclusions.csv", index=False)

# ============================================================
# 5. Print row counts to confirm expectations
# ============================================================

expected_appendix_rows = 2052 + 3 + 1  # 2052 cancer + 3 delirium ICD + 1 documentation note
expected_exclusion_rows = total_cancer_candidates - 2052

print()
print("=" * 55)
print("FILE ROW COUNT CONFIRMATION")
print("=" * 55)
print(f"icd_code_appendix.csv: {len(appendix_table)} rows "
      f"(expected 2052 cancer + 3 delirium ICD + 1 note = {expected_appendix_rows})")
print(f"  Match: {len(appendix_table) == expected_appendix_rows}")
print()
print(f"icd_code_exclusions.csv: {len(excluded_df)} rows "
      f"(expected {total_cancer_candidates} candidates - 2052 kept = {expected_exclusion_rows})")
print(f"  Match: {len(excluded_df) == expected_exclusion_rows}")
print()
print("Exclusion reason breakdown:")
print(excluded_df['exclusion_reason'].str.split(':').str[0].value_counts())

In [ ]:
import pandas as pd
import re

raw_path = "../data/raw/"
processed_path = "../data/processed/"

# ============================================================
# Reload everything fresh
# ============================================================

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])

d_icd_diagnoses = pd.read_csv(raw_path + "d_icd_diagnoses.csv.gz", compression="gzip")
d_icd_diagnoses['icd_code'] = d_icd_diagnoses['icd_code'].astype(str).str.strip()

diagnoses_icd = pd.read_csv(raw_path + "diagnoses_icd.csv.gz", compression="gzip")
diagnoses_icd['icd_code'] = diagnoses_icd['icd_code'].astype(str).str.strip()

print(f"cohort_step1.csv loaded: {cohort_step1.shape}")
print()

# ============================================================
# 1. icu_admission_year
# ============================================================

cohort_step1['icu_admission_year'] = cohort_step1['intime'].dt.year

# ============================================================
# Rebuild the confirmed 2052-code cancer list (full-table-scan version)
# ============================================================

def icd9_category(code):
    code = str(code)
    return int(code[:3]) if code[:3].isdigit() else None

def icd10_cancer_match(code):
    m = re.match(r'^C(\d{2})', str(code))
    return 0 <= int(m.group(1)) <= 96 if m else False

def cancer_range_match(row):
    code = str(row['icd_code'])
    if row['icd_version'] == 9:
        cat = icd9_category(code)
        if cat is None:
            return False
        return (140 <= cat <= 172) or (174 <= cat <= 189) or (190 <= cat <= 209)
    elif row['icd_version'] == 10:
        return icd10_cancer_match(code)
    return False

def is_carcinoma_in_situ(row):
    code = str(row['icd_code'])
    if row['icd_version'] == 9:
        cat = icd9_category(code)
        return cat is not None and 230 <= cat <= 234
    elif row['icd_version'] == 10:
        return code[:2] == 'D0'
    return False

exclude_terms = [
    'in remission', 'personal history', 'family history', 'screening',
    'genetic susceptibility', 'follow-up', 'prophylactic',
    'complicating pregnancy', 'complicating childbirth', 'complicating the puerperium'
]
exclude_pattern = '|'.join(exclude_terms)
explicit_exclude_codes = {'79582', 'R971', 'R9721', 'D701'}

range_mask = d_icd_diagnoses.apply(cancer_range_match, axis=1)
term_mask = d_icd_diagnoses['long_title'].str.contains(exclude_pattern, case=False, na=False, regex=True)
explicit_mask = d_icd_diagnoses['icd_code'].isin(explicit_exclude_codes)
cis_mask = d_icd_diagnoses.apply(is_carcinoma_in_situ, axis=1)

cancer_include_mask = range_mask & ~term_mask & ~explicit_mask & ~cis_mask
cancer_final_codes = set(d_icd_diagnoses.loc[cancer_include_mask, 'icd_code'])

print(f"Cancer codes confirmed: {len(cancer_final_codes)} (expected 2052)")
print()

# ============================================================
# Get all cancer diagnosis rows (icd_code + long_title) for cohort's hadm_id
# ============================================================

cohort_hadm_ids = set(cohort_step1['hadm_id'])

cancer_diag_rows = diagnoses_icd[
    diagnoses_icd['hadm_id'].isin(cohort_hadm_ids) &
    diagnoses_icd['icd_code'].isin(cancer_final_codes)
].merge(
    d_icd_diagnoses[['icd_code', 'icd_version', 'long_title']],
    on='icd_code', how='left', suffixes=('', '_lookup')
)

print(f"Cancer diagnosis rows found for cohort patients: {len(cancer_diag_rows)}")
print(f"hadm_id with at least one cancer diagnosis: {cancer_diag_rows['hadm_id'].nunique()} "
      f"(should equal {len(cohort_hadm_ids)} if every cohort hadm_id is cancer-positive)")
print()

# ============================================================
# 2. Classification rules — keyword-based on long_title
# ============================================================

def classify_broad(title):
    """OPTION A: Binary Hematologic vs Solid tumor."""
    title_l = str(title).lower()
    heme_terms = ['leukemia', 'lymphoma', 'myeloma', 'hodgkin', "waldeyer",
                  'lymphoid', 'myeloid', 'monocytic', 'erythroleukemia',
                  'plasma cell', 'mast cell leukemia']
    if any(term in title_l for term in heme_terms):
        return 'Hematologic'
    return 'Solid tumor'

def classify_granular(title):
    """OPTION B: Organ-system level breakdown."""
    title_l = str(title).lower()

    heme_terms = ['leukemia', 'lymphoma', 'myeloma', 'hodgkin', "waldeyer",
                  'lymphoid', 'myeloid', 'monocytic', 'erythroleukemia',
                  'plasma cell', 'mast cell leukemia']
    if any(term in title_l for term in heme_terms):
        return 'Hematologic'

    if 'breast' in title_l:
        return 'Breast'

    resp_terms = ['lung', 'bronchus', 'trachea', 'pleura', 'mediastinum']
    if any(term in title_l for term in resp_terms):
        return 'Respiratory/Thoracic'

    headneck_terms = ['lip', 'tongue', 'mouth', 'pharynx', 'larynx',
                       'salivary', 'nasal', 'sinus', 'nasopharynx',
                       'oropharynx', 'hypopharynx', 'tonsil', 'gum', 'palate']
    if any(term in title_l for term in headneck_terms):
        return 'Head and Neck'

    gi_terms = ['esophagus', 'stomach', 'colon', 'rectum', 'liver',
                'pancrea', 'gallbladder', 'intestine', 'bile duct',
                'anus', 'anal canal', 'peritoneum', 'digestive']
    if any(term in title_l for term in gi_terms):
        return 'Gastrointestinal'

    gu_terms = ['kidney', 'bladder', 'prostate', 'testis', 'penis',
                'ureter', 'urethra', 'urinary', 'renal pelvis']
    if any(term in title_l for term in gu_terms):
        return 'Genitourinary'

    gyn_terms = ['cervix', 'uterus', 'uterine', 'ovary', 'ovarian',
                 'vagina', 'vulva', 'fallopian', 'endometri']
    if any(term in title_l for term in gyn_terms):
        return 'Gynecologic'

    cns_terms = ['brain', 'meninges', 'spinal cord', 'cerebrum',
                 'cerebellum', 'nervous system', 'cranial nerve']
    if any(term in title_l for term in cns_terms):
        return 'CNS/Brain'

    skin_terms = ['skin', 'melanoma', 'merkel cell']
    if any(term in title_l for term in skin_terms):
        return 'Skin'

    endo_terms = ['thyroid', 'adrenal', 'pituitary', 'parathyroid',
                  'endocrine', 'pineal', 'carotid body']
    if any(term in title_l for term in endo_terms):
        return 'Endocrine'

    bone_terms = ['bone', 'cartilage', 'connective', 'soft tissue']
    if any(term in title_l for term in bone_terms):
        return 'Bone/Soft Tissue'

    return 'Other/Unspecified'

cancer_diag_rows['category_A'] = cancer_diag_rows['long_title'].apply(classify_broad)
cancer_diag_rows['category_B'] = cancer_diag_rows['long_title'].apply(classify_granular)

# ============================================================
# Aggregate to one category per hadm_id
# Priority: Hematologic first (if ANY diagnosis is hematologic, patient counts as hematologic
# for cancer-focused subgroup analysis relevance) — otherwise most frequent solid-tumor category
# for that patient; ties broken by first occurrence (seq_num order, already default row order).
# ============================================================

def aggregate_patient_category(sub_df, col):
    cats = sub_df[col].tolist()
    if 'Hematologic' in cats:
        return 'Hematologic'
    # otherwise take the most frequent non-hematologic category for this patient
    return pd.Series(cats).value_counts().idxmax()

patient_category_A = cancer_diag_rows.groupby('hadm_id').apply(
    lambda g: aggregate_patient_category(g, 'category_A')
).rename('cancer_type_optionA')

patient_category_B = cancer_diag_rows.groupby('hadm_id').apply(
    lambda g: aggregate_patient_category(g, 'category_B')
).rename('cancer_type_optionB')

# ============================================================
# Show counts for BOTH options
# ============================================================

print("=" * 55)
print("OPTION A: Hematologic vs Solid tumor")
print("=" * 55)
print(patient_category_A.value_counts())
print()

print("=" * 55)
print("OPTION B: Granular organ-system breakdown")
print("=" * 55)
print(patient_category_B.value_counts())
print()

# ============================================================
# 3. delirium_subtype NOT added — confirmed, pending RASS coverage check
# ============================================================

print("NOTE: delirium_subtype NOT added — pending RASS coverage check (Notebook 1, probable addition).")
print()

# ============================================================
# 4. Merge icu_admission_year (already added) — cancer_type NOT saved yet,
#    pending your choice of Option A vs Option B
# ============================================================

cohort_step1_preview = cohort_step1.merge(
    patient_category_A.reset_index(), on='hadm_id', how='left'
).merge(
    patient_category_B.reset_index(), on='hadm_id', how='left'
)

print(f"Preview shape with both category options attached: {cohort_step1_preview.shape}")
print(f"Any hadm_id missing a cancer category (should be 0): "
      f"{cohort_step1_preview['cancer_type_optionA'].isna().sum()}")

In [ ]:
import pandas as pd
import re

raw_path = "../data/raw/"
processed_path = "../data/processed/"

# ============================================================
# Reload everything fresh
# ============================================================

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])

d_icd_diagnoses = pd.read_csv(raw_path + "d_icd_diagnoses.csv.gz", compression="gzip")
d_icd_diagnoses['icd_code'] = d_icd_diagnoses['icd_code'].astype(str).str.strip()

diagnoses_icd = pd.read_csv(raw_path + "diagnoses_icd.csv.gz", compression="gzip")
diagnoses_icd['icd_code'] = diagnoses_icd['icd_code'].astype(str).str.strip()

print(f"cohort_step1.csv loaded: {cohort_step1.shape}")

# ============================================================
# 1. icu_admission_year
# ============================================================

cohort_step1['icu_admission_year'] = cohort_step1['intime'].dt.year

# ============================================================
# Rebuild the confirmed 2052-code cancer list
# ============================================================

def icd9_category(code):
    code = str(code)
    return int(code[:3]) if code[:3].isdigit() else None

def icd10_cancer_match(code):
    m = re.match(r'^C(\d{2})', str(code))
    return 0 <= int(m.group(1)) <= 96 if m else False

def cancer_range_match(row):
    code = str(row['icd_code'])
    if row['icd_version'] == 9:
        cat = icd9_category(code)
        if cat is None:
            return False
        return (140 <= cat <= 172) or (174 <= cat <= 189) or (190 <= cat <= 209)
    elif row['icd_version'] == 10:
        return icd10_cancer_match(code)
    return False

def is_carcinoma_in_situ(row):
    code = str(row['icd_code'])
    if row['icd_version'] == 9:
        cat = icd9_category(code)
        return cat is not None and 230 <= cat <= 234
    elif row['icd_version'] == 10:
        return code[:2] == 'D0'
    return False

exclude_terms = [
    'in remission', 'personal history', 'family history', 'screening',
    'genetic susceptibility', 'follow-up', 'prophylactic',
    'complicating pregnancy', 'complicating childbirth', 'complicating the puerperium'
]
exclude_pattern = '|'.join(exclude_terms)
explicit_exclude_codes = {'79582', 'R971', 'R9721', 'D701'}

range_mask = d_icd_diagnoses.apply(cancer_range_match, axis=1)
term_mask = d_icd_diagnoses['long_title'].str.contains(exclude_pattern, case=False, na=False, regex=True)
explicit_mask = d_icd_diagnoses['icd_code'].isin(explicit_exclude_codes)
cis_mask = d_icd_diagnoses.apply(is_carcinoma_in_situ, axis=1)

cancer_include_mask = range_mask & ~term_mask & ~explicit_mask & ~cis_mask
cancer_final_codes = set(d_icd_diagnoses.loc[cancer_include_mask, 'icd_code'])

print(f"Cancer codes confirmed: {len(cancer_final_codes)} (expected 2052)")

# ============================================================
# Get cancer diagnosis rows for cohort's hadm_id
# ============================================================

cohort_hadm_ids = set(cohort_step1['hadm_id'])

cancer_diag_rows = diagnoses_icd[
    diagnoses_icd['hadm_id'].isin(cohort_hadm_ids) &
    diagnoses_icd['icd_code'].isin(cancer_final_codes)
].merge(
    d_icd_diagnoses[['icd_code', 'icd_version', 'long_title']],
    on='icd_code', how='left'
)

print(f"Cancer diagnosis rows found: {len(cancer_diag_rows)}")
print()

# ============================================================
# 2. Option B classification (organ-system level)
# ============================================================

def classify_granular(title):
    title_l = str(title).lower()

    heme_terms = ['leukemia', 'lymphoma', 'myeloma', 'hodgkin', "waldeyer",
                  'lymphoid', 'myeloid', 'monocytic', 'erythroleukemia',
                  'plasma cell', 'mast cell leukemia']
    if any(term in title_l for term in heme_terms):
        return 'Hematologic'

    if 'breast' in title_l:
        return 'Breast'

    resp_terms = ['lung', 'bronchus', 'trachea', 'pleura', 'mediastinum']
    if any(term in title_l for term in resp_terms):
        return 'Respiratory/Thoracic'

    headneck_terms = ['lip', 'tongue', 'mouth', 'pharynx', 'larynx',
                       'salivary', 'nasal', 'sinus', 'nasopharynx',
                       'oropharynx', 'hypopharynx', 'tonsil', 'gum', 'palate']
    if any(term in title_l for term in headneck_terms):
        return 'Head and Neck'

    gi_terms = ['esophagus', 'stomach', 'colon', 'rectum', 'liver',
                'pancrea', 'gallbladder', 'intestine', 'bile duct',
                'anus', 'anal canal', 'peritoneum', 'digestive']
    if any(term in title_l for term in gi_terms):
        return 'Gastrointestinal'

    gu_terms = ['kidney', 'bladder', 'prostate', 'testis', 'penis',
                'ureter', 'urethra', 'urinary', 'renal pelvis']
    if any(term in title_l for term in gu_terms):
        return 'Genitourinary'

    gyn_terms = ['cervix', 'uterus', 'uterine', 'ovary', 'ovarian',
                 'vagina', 'vulva', 'fallopian', 'endometri']
    if any(term in title_l for term in gyn_terms):
        return 'Gynecologic'

    cns_terms = ['brain', 'meninges', 'spinal cord', 'cerebrum',
                 'cerebellum', 'nervous system', 'cranial nerve']
    if any(term in title_l for term in cns_terms):
        return 'CNS/Brain'

    skin_terms = ['skin', 'melanoma', 'merkel cell']
    if any(term in title_l for term in skin_terms):
        return 'Skin'

    endo_terms = ['thyroid', 'adrenal', 'pituitary', 'parathyroid',
                  'endocrine', 'pineal', 'carotid body']
    if any(term in title_l for term in endo_terms):
        return 'Endocrine'

    bone_terms = ['bone', 'cartilage', 'connective', 'soft tissue']
    if any(term in title_l for term in bone_terms):
        return 'Bone/Soft Tissue'

    return 'Other/Unspecified'

cancer_diag_rows['category_B'] = cancer_diag_rows['long_title'].apply(classify_granular)

def aggregate_patient_category(sub_df, col):
    cats = sub_df[col].tolist()
    if 'Hematologic' in cats:
        return 'Hematologic'
    return pd.Series(cats).value_counts().idxmax()

patient_category_B = cancer_diag_rows.groupby('hadm_id').apply(
    lambda g: aggregate_patient_category(g, 'category_B')
).rename('cancer_type_raw').reset_index()

print("Category counts BEFORE merging small groups:")
print(patient_category_B['cancer_type_raw'].value_counts())
print()

# ============================================================
# Merge the 5 small categories into "Other/Unspecified"
# ============================================================

merge_into_other = {'Head and Neck', 'Gynecologic', 'Breast', 'Endocrine', 'Skin'}

patient_category_B['cancer_type'] = patient_category_B['cancer_type_raw'].apply(
    lambda cat: 'Other/Unspecified' if cat in merge_into_other else cat
)

expected_final_categories = {
    'Gastrointestinal', 'Hematologic', 'Respiratory/Thoracic',
    'Other/Unspecified', 'Genitourinary', 'CNS/Brain', 'Bone/Soft Tissue'
}
actual_categories = set(patient_category_B['cancer_type'].unique())

print(f"Final category set matches expected 7: {actual_categories == expected_final_categories}")
if actual_categories != expected_final_categories:
    print(f"  Expected: {sorted(expected_final_categories)}")
    print(f"  Actual:   {sorted(actual_categories)}")
print()

# ============================================================
# 4. Merge into cohort_step1, drop raw/intermediate columns, save
# ============================================================

cohort_step1 = cohort_step1.merge(
    patient_category_B[['hadm_id', 'cancer_type']], on='hadm_id', how='left'
)

missing_category = cohort_step1['cancer_type'].isna().sum()
print(f"Rows missing cancer_type after merge (should be 0): {missing_category}")

cohort_step1.to_csv(processed_path + "cohort_step1.csv", index=False)

print()
print(f"Saved cohort_step1.csv with icu_admission_year and cancer_type columns.")
print(f"Final shape: {cohort_step1.shape}")
print()
print("Final cancer_type value_counts():")
print(cohort_step1['cancer_type'].value_counts())

In [ ]:
import pandas as pd

raw_path = "../data/raw/"
processed_path = "../data/processed/"

# ============================================================
# 1. Confirm RASS itemid from d_items — do NOT assume 228096
# ============================================================

d_items = pd.read_csv(raw_path + "d_items.csv.gz", compression="gzip")

rass_items = d_items[
    d_items['label'].str.contains('Richmond', case=False, na=False) |
    d_items['label'].str.contains('RASS', case=False, na=False)
].copy()

print(f"RASS/Richmond related items found in d_items: {len(rass_items)}")
print(rass_items.to_string(index=False))
print()
print("STOP AND CONFIRM: check the printed itemid(s) above against the label text")
print("before trusting the itemid list built below.")
print()

rass_itemids = set(rass_items['itemid'])

# ============================================================
# Load cohort_step1.csv
# ============================================================

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
cohort_step1['outtime'] = pd.to_datetime(cohort_step1['outtime'])

print(f"cohort_step1.csv loaded: {cohort_step1.shape}")

cohort_subject_ids = set(cohort_step1['subject_id'])
print(f"Unique subject_id in cohort: {len(cohort_subject_ids)}")
print()

# ============================================================
# 2. Chunk-read chartevents, filtered to RASS itemids + cohort subject_ids
# ============================================================

chunk_size = 1_000_000
rass_rows = []
usecols = ['subject_id', 'hadm_id', 'itemid', 'charttime', 'value', 'valuenum']

for chunk in pd.read_csv(
    raw_path + "chartevents.csv.gz",
    compression="gzip",
    usecols=usecols,
    chunksize=chunk_size
):
    filtered = chunk[
        chunk['itemid'].isin(rass_itemids) &
        chunk['subject_id'].isin(cohort_subject_ids)
    ]
    if len(filtered) > 0:
        rass_rows.append(filtered)

rass_events = pd.concat(rass_rows, ignore_index=True) if rass_rows else pd.DataFrame(columns=usecols)
rass_events['charttime'] = pd.to_datetime(rass_events['charttime'])

print(f"RASS chartevents rows pulled (cohort subject_ids only): {len(rass_events)}")
print()

# ============================================================
# Restrict each patient's RASS events to their OWN ICU stay window (intime-outtime)
# ============================================================

rass_events = rass_events.merge(
    cohort_step1[['subject_id', 'intime', 'outtime']],
    on='subject_id', how='left'
)

rass_events_in_stay = rass_events[
    (rass_events['charttime'] >= rass_events['intime']) &
    (rass_events['charttime'] <= rass_events['outtime'])
].copy()

print(f"RASS events falling within each patient's actual ICU stay window: {len(rass_events_in_stay)}")
print()

# ============================================================
# 3a. % of cohort with at least 1 RASS value during their stay
# ============================================================

subjects_with_rass = set(rass_events_in_stay['subject_id'])
n_with_rass = len(subjects_with_rass)
pct_with_rass = 100 * n_with_rass / len(cohort_step1)

print(f"Patients with >=1 RASS value during ICU stay: {n_with_rass} / {len(cohort_step1)} ({pct_with_rass:.2f}%)")
print()

# ============================================================
# 3b. % with RASS charted at least every 4 hours during first 24h
# Bin the first 24h (or the full stay if shorter than 24h) into 4h windows;
# require every window to have >=1 reading.
# ============================================================

def check_4h_coverage(row, events_df):
    subj_events = events_df[events_df['subject_id'] == row['subject_id']]
    window_end = min(row['intime'] + pd.Timedelta(hours=24), row['outtime'])
    window_start = row['intime']
    stay_duration_hours = (window_end - window_start).total_seconds() / 3600

    if stay_duration_hours <= 0:
        return False, 0  # degenerate stay, cannot evaluate

    n_bins = max(1, int(stay_duration_hours // 4) + (1 if stay_duration_hours % 4 > 0 else 0))

    bin_covered = []
    for i in range(n_bins):
        bin_start = window_start + pd.Timedelta(hours=4*i)
        bin_end = min(window_start + pd.Timedelta(hours=4*(i+1)), window_end)
        has_reading = ((subj_events['charttime'] >= bin_start) & (subj_events['charttime'] < bin_end)).any()
        bin_covered.append(has_reading)

    return all(bin_covered), n_bins

coverage_results = cohort_step1.apply(
    lambda row: check_4h_coverage(row, rass_events_in_stay), axis=1
)

cohort_step1['rass_4h_coverage'] = coverage_results.apply(lambda x: x[0])

n_4h_covered = cohort_step1['rass_4h_coverage'].sum()
pct_4h_covered = 100 * n_4h_covered / len(cohort_step1)

print(f"Patients with RASS charted at least every 4h during first 24h "
      f"(or full stay if shorter): {n_4h_covered} / {len(cohort_step1)} ({pct_4h_covered:.2f}%)")
print("  NOTE: for patients whose ICU stay was shorter than 24h, the coverage window was")
print("  clipped to their actual outtime (can't chart after discharge) — this is a deliberate")
print("  adjustment, not a bug, and is why the denominator logic isn't a flat 6-bins-per-patient rule.")
print()

# ============================================================
# 3c. Distribution of RASS readings per patient
# ============================================================

readings_per_patient = rass_events_in_stay.groupby('subject_id').size()

# Include patients with ZERO readings in the distribution (they're part of the cohort too)
readings_per_patient_full = readings_per_patient.reindex(cohort_step1['subject_id'], fill_value=0)

print("RASS readings per patient — distribution (including 0-reading patients):")
print(f"  Min:    {readings_per_patient_full.min()}")
print(f"  Median: {readings_per_patient_full.median()}")
print(f"  Max:    {readings_per_patient_full.max()}")
print()

# ============================================================
# 4. Clean summary table
# ============================================================

summary = pd.DataFrame([
    {"Metric": "Cohort size", "Value": len(cohort_step1)},
    {"Metric": "RASS itemid(s) used", "Value": sorted(rass_itemids)},
    {"Metric": "Patients with >=1 RASS value in stay", "Value": f"{n_with_rass} ({pct_with_rass:.2f}%)"},
    {"Metric": "Patients with RASS every 4h in first 24h", "Value": f"{n_4h_covered} ({pct_4h_covered:.2f}%)"},
    {"Metric": "RASS readings/patient — min", "Value": readings_per_patient_full.min()},
    {"Metric": "RASS readings/patient — median", "Value": readings_per_patient_full.median()},
    {"Metric": "RASS readings/patient — max", "Value": readings_per_patient_full.max()},
])

print("=" * 55)
print("SUMMARY TABLE")
print("=" * 55)
print(summary.to_string(index=False))

In [ ]:
import pandas as pd

raw_path = "../data/raw/"
processed_path = "../data/processed/"

RASS_ITEMID = 228096  # Richmond-RAS Scale, confirmed — excludes 228299 (goal/target) and 228302 (CAM-ICU gating)

# ============================================================
# Load cohort_step1.csv
# ============================================================

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
cohort_step1['outtime'] = pd.to_datetime(cohort_step1['outtime'])

print(f"cohort_step1.csv loaded: {cohort_step1.shape}")

cohort_subject_ids = set(cohort_step1['subject_id'])
print(f"Unique subject_id in cohort: {len(cohort_subject_ids)}")
print()

# ============================================================
# 1. Chunk-read chartevents, filtered to itemid 228096 + cohort subject_ids
# ============================================================

chunk_size = 1_000_000
rass_rows = []
usecols = ['subject_id', 'hadm_id', 'itemid', 'charttime', 'value', 'valuenum']

for chunk in pd.read_csv(
    raw_path + "chartevents.csv.gz",
    compression="gzip",
    usecols=usecols,
    chunksize=chunk_size
):
    filtered = chunk[
        (chunk['itemid'] == RASS_ITEMID) &
        chunk['subject_id'].isin(cohort_subject_ids)
    ]
    if len(filtered) > 0:
        rass_rows.append(filtered)

rass_events = pd.concat(rass_rows, ignore_index=True) if rass_rows else pd.DataFrame(columns=usecols)
rass_events['charttime'] = pd.to_datetime(rass_events['charttime'])

print(f"RASS (itemid {RASS_ITEMID}) chartevents rows pulled (cohort subject_ids only): {len(rass_events)}")
print()

# ============================================================
# 2. Restrict to each patient's own ICU stay window (intime-outtime)
# ============================================================

rass_events = rass_events.merge(
    cohort_step1[['subject_id', 'intime', 'outtime']],
    on='subject_id', how='left'
)

rass_events_in_stay = rass_events[
    (rass_events['charttime'] >= rass_events['intime']) &
    (rass_events['charttime'] <= rass_events['outtime'])
].copy()

print(f"RASS events falling within each patient's actual ICU stay: {len(rass_events_in_stay)}")
print()

# First-24h subset
rass_events_in_stay['first_24h_cutoff'] = rass_events_in_stay['intime'] + pd.Timedelta(hours=24)
rass_events_first24h = rass_events_in_stay[
    rass_events_in_stay['charttime'] <= rass_events_in_stay['first_24h_cutoff']
].copy()

# ============================================================
# 3a. % with at least 1 RASS value during entire stay
# ============================================================

subjects_any_rass = set(rass_events_in_stay['subject_id'])
n_any_rass = len(subjects_any_rass)
pct_any_rass = 100 * n_any_rass / len(cohort_step1)

# ============================================================
# 3b. % with at least 1 RASS value in first 24h
# ============================================================

subjects_rass_24h = set(rass_events_first24h['subject_id'])
n_rass_24h = len(subjects_rass_24h)
pct_rass_24h = 100 * n_rass_24h / len(cohort_step1)

# ============================================================
# 3c. % with >=6 RASS readings in first 24h (roughly every 4h or more often)
# ============================================================

readings_first24h_count = rass_events_first24h.groupby('subject_id').size()
readings_first24h_full = readings_first24h_count.reindex(cohort_step1['subject_id'], fill_value=0)

n_frequent = (readings_first24h_full >= 6).sum()
pct_frequent = 100 * n_frequent / len(cohort_step1)

# ============================================================
# 3d. Distribution of total RASS readings per patient (full stay)
# ============================================================

readings_per_patient = rass_events_in_stay.groupby('subject_id').size()
readings_per_patient_full = readings_per_patient.reindex(cohort_step1['subject_id'], fill_value=0)

dist_min = readings_per_patient_full.min()
dist_p25 = readings_per_patient_full.quantile(0.25)
dist_median = readings_per_patient_full.median()
dist_p75 = readings_per_patient_full.quantile(0.75)
dist_max = readings_per_patient_full.max()

# ============================================================
# 4. Clean summary table
# ============================================================

summary = pd.DataFrame([
    {"Metric": "Cohort size", "Value": len(cohort_step1)},
    {"Metric": "RASS itemid used", "Value": RASS_ITEMID},
    {"Metric": "Patients with >=1 RASS value (entire stay)", "Value": f"{n_any_rass} ({pct_any_rass:.2f}%)"},
    {"Metric": "Patients with >=1 RASS value (first 24h)", "Value": f"{n_rass_24h} ({pct_rass_24h:.2f}%)"},
    {"Metric": "Patients with >=6 RASS readings in first 24h (~q4h or more often)", "Value": f"{n_frequent} ({pct_frequent:.2f}%)"},
    {"Metric": "RASS readings/patient (full stay) — min", "Value": dist_min},
    {"Metric": "RASS readings/patient (full stay) — 25th pct", "Value": dist_p25},
    {"Metric": "RASS readings/patient (full stay) — median", "Value": dist_median},
    {"Metric": "RASS readings/patient (full stay) — 75th pct", "Value": dist_p75},
    {"Metric": "RASS readings/patient (full stay) — max", "Value": dist_max},
])

print("=" * 55)
print("RASS COVERAGE SUMMARY (itemid 228096 only)")
print("=" * 55)
print(summary.to_string(index=False))

In [ ]:
import pandas as pd

raw_path = "../data/raw/"
processed_path = "../data/processed/"

RASS_ITEMID = 228096

# ============================================================
# Reload cohort_step1.csv and re-pull RASS events fresh (self-contained,
# not relying on session state from the previous cell)
# ============================================================

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
cohort_step1['outtime'] = pd.to_datetime(cohort_step1['outtime'])

print(f"cohort_step1.csv loaded: {cohort_step1.shape}")
cohort_subject_ids = set(cohort_step1['subject_id'])

chunk_size = 1_000_000
rass_rows = []
usecols = ['subject_id', 'hadm_id', 'itemid', 'charttime', 'value', 'valuenum']

for chunk in pd.read_csv(
    raw_path + "chartevents.csv.gz",
    compression="gzip",
    usecols=usecols,
    chunksize=chunk_size
):
    filtered = chunk[
        (chunk['itemid'] == RASS_ITEMID) &
        chunk['subject_id'].isin(cohort_subject_ids)
    ]
    if len(filtered) > 0:
        rass_rows.append(filtered)

rass_events = pd.concat(rass_rows, ignore_index=True) if rass_rows else pd.DataFrame(columns=usecols)
rass_events['charttime'] = pd.to_datetime(rass_events['charttime'])

rass_events = rass_events.merge(
    cohort_step1[['subject_id', 'intime', 'outtime']],
    on='subject_id', how='left'
)

rass_events_in_stay = rass_events[
    (rass_events['charttime'] >= rass_events['intime']) &
    (rass_events['charttime'] <= rass_events['outtime'])
].copy()

print(f"RASS events pulled within ICU stays: {len(rass_events_in_stay)} (expected 92046)")
print()

# ============================================================
# Check valuenum is populated and numeric before trusting sign logic
# ============================================================

n_missing_valuenum = rass_events_in_stay['valuenum'].isna().sum()
print(f"RASS rows missing valuenum: {n_missing_valuenum} out of {len(rass_events_in_stay)}")
if n_missing_valuenum > 0:
    print("  These rows will be excluded from classification (can't determine sign without a numeric value).")
    print("  Sample of rows with missing valuenum (checking 'value' text field instead):")
    print(rass_events_in_stay[rass_events_in_stay['valuenum'].isna()][['value', 'valuenum']].head(10))
print()

rass_events_valid = rass_events_in_stay.dropna(subset=['valuenum']).copy()

print(f"RASS score range observed: min={rass_events_valid['valuenum'].min()}, "
      f"max={rass_events_valid['valuenum'].max()}")
print()

# ============================================================
# 1. Classify delirium_subtype per patient
# ============================================================

def classify_subtype(sub_df):
    if len(sub_df) == 0:
        return 'Unknown'
    has_positive = (sub_df['valuenum'] >= 1).any()
    has_nonpositive = (sub_df['valuenum'] <= 0).any()
    if has_positive and has_nonpositive:
        return 'Mixed'
    elif has_positive:
        return 'Hyperactive'
    elif has_nonpositive:
        return 'Hypoactive'
    else:
        return 'Unknown'  # shouldn't occur given the two conditions above are exhaustive

patient_subtype = rass_events_valid.groupby('subject_id').apply(classify_subtype).rename('delirium_subtype')

# Patients with zero valid RASS readings in their stay -> Unknown
patient_subtype_full = patient_subtype.reindex(cohort_step1['subject_id'])
patient_subtype_full = patient_subtype_full.fillna('Unknown')

n_unknown = (patient_subtype_full == 'Unknown').sum()
print(f"Patients classified as Unknown (zero valid RASS readings in stay): {n_unknown} (expected ~73)")
print()

# ============================================================
# 2. value_counts()
# ============================================================

print("=" * 55)
print("delirium_subtype value_counts()")
print("=" * 55)
print(patient_subtype_full.value_counts())
print()

# ============================================================
# 3. Add to cohort_step1.csv, save, confirm shape
# ============================================================

cohort_step1 = cohort_step1.merge(
    patient_subtype_full.rename('delirium_subtype').reset_index(),
    on='subject_id', how='left'
)

cohort_step1.to_csv(processed_path + "cohort_step1.csv", index=False)

print(f"Saved cohort_step1.csv with delirium_subtype column.")
print(f"Final shape: {cohort_step1.shape} (expected (3122, 11))")

In [ ]:
import pandas as pd

raw_path = "../data/raw/"
processed_path = "../data/processed/"

RASS_ITEMID = 228096

# ============================================================
# Reload cohort_step1.csv and re-pull RASS events fresh (self-contained,
# not relying on session state from the previous cell)
# ============================================================

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
cohort_step1['outtime'] = pd.to_datetime(cohort_step1['outtime'])

print(f"cohort_step1.csv loaded: {cohort_step1.shape}")
cohort_subject_ids = set(cohort_step1['subject_id'])

chunk_size = 1_000_000
rass_rows = []
usecols = ['subject_id', 'hadm_id', 'itemid', 'charttime', 'value', 'valuenum']

for chunk in pd.read_csv(
    raw_path + "chartevents.csv.gz",
    compression="gzip",
    usecols=usecols,
    chunksize=chunk_size
):
    filtered = chunk[
        (chunk['itemid'] == RASS_ITEMID) &
        chunk['subject_id'].isin(cohort_subject_ids)
    ]
    if len(filtered) > 0:
        rass_rows.append(filtered)

rass_events = pd.concat(rass_rows, ignore_index=True) if rass_rows else pd.DataFrame(columns=usecols)
rass_events['charttime'] = pd.to_datetime(rass_events['charttime'])

rass_events = rass_events.merge(
    cohort_step1[['subject_id', 'intime', 'outtime']],
    on='subject_id', how='left'
)

rass_events_in_stay = rass_events[
    (rass_events['charttime'] >= rass_events['intime']) &
    (rass_events['charttime'] <= rass_events['outtime'])
].copy()

print(f"RASS events pulled within ICU stays: {len(rass_events_in_stay)} (expected 92046)")
print()

# ============================================================
# Check valuenum is populated and numeric before trusting sign logic
# ============================================================

n_missing_valuenum = rass_events_in_stay['valuenum'].isna().sum()
print(f"RASS rows missing valuenum: {n_missing_valuenum} out of {len(rass_events_in_stay)}")
if n_missing_valuenum > 0:
    print("  These rows will be excluded from classification (can't determine sign without a numeric value).")
    print("  Sample of rows with missing valuenum (checking 'value' text field instead):")
    print(rass_events_in_stay[rass_events_in_stay['valuenum'].isna()][['value', 'valuenum']].head(10))
print()

rass_events_valid = rass_events_in_stay.dropna(subset=['valuenum']).copy()

print(f"RASS score range observed: min={rass_events_valid['valuenum'].min()}, "
      f"max={rass_events_valid['valuenum'].max()}")
print()

# ============================================================
# 1. Classify delirium_subtype per patient
# ============================================================

def classify_subtype(sub_df):
    if len(sub_df) == 0:
        return 'Unknown'
    has_positive = (sub_df['valuenum'] >= 1).any()
    has_nonpositive = (sub_df['valuenum'] <= 0).any()
    if has_positive and has_nonpositive:
        return 'Mixed'
    elif has_positive:
        return 'Hyperactive'
    elif has_nonpositive:
        return 'Hypoactive'
    else:
        return 'Unknown'  # shouldn't occur given the two conditions above are exhaustive

patient_subtype = rass_events_valid.groupby('subject_id').apply(classify_subtype).rename('delirium_subtype')

# Patients with zero valid RASS readings in their stay -> Unknown
patient_subtype_full = patient_subtype.reindex(cohort_step1['subject_id'])
patient_subtype_full = patient_subtype_full.fillna('Unknown')

n_unknown = (patient_subtype_full == 'Unknown').sum()
print(f"Patients classified as Unknown (zero valid RASS readings in stay): {n_unknown} (expected ~73)")
print()

# ============================================================
# 2. value_counts()
# ============================================================

print("=" * 55)
print("delirium_subtype value_counts()")
print("=" * 55)
print(patient_subtype_full.value_counts())
print()

# ============================================================
# 3. Add to cohort_step1.csv, save, confirm shape
# ============================================================

cohort_step1 = cohort_step1.merge(
    patient_subtype_full.rename('delirium_subtype').reset_index(),
    on='subject_id', how='left'
)

cohort_step1.to_csv(processed_path + "cohort_step1.csv", index=False)

print(f"Saved cohort_step1.csv with delirium_subtype column.")
print(f"Final shape: {cohort_step1.shape} (expected (3122, 11))")

In [ ]:
import pandas as pd

raw_path = "../data/raw/"
processed_path = "../data/processed/"

# ============================================================
# Reload cohort_step1.csv fresh
# ============================================================

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
cohort_step1['outtime'] = pd.to_datetime(cohort_step1['outtime'])

print(f"cohort_step1.csv loaded: {cohort_step1.shape}")
cohort_hadm_ids = set(cohort_step1['hadm_id'])
cohort_subject_ids = set(cohort_step1['subject_id'])
print()

# ============================================================
# Confirmed CAM-ICU feature mapping (do not guess — as given)
# ============================================================

feature_map = {
    228300: 'ms_change', 228337: 'ms_change', 229326: 'ms_change',
    228301: 'inattention', 228336: 'inattention', 229325: 'inattention',
    228303: 'disorganized_thinking', 228335: 'disorganized_thinking', 229324: 'disorganized_thinking',
    228334: 'altered_loc',
    # 228302 (RASS LOC) deliberately excluded — gating item, not diagnostic
}
cam_icu_itemids = set(feature_map.keys())
RASS_ITEMID = 228096

all_relevant_itemids = cam_icu_itemids | {RASS_ITEMID}

# ============================================================
# Chunk-read chartevents ONCE for both CAM-ICU features and RASS,
# filtered to cohort's hadm_id (all cohort patients are already cancer-positive)
# ============================================================

chunk_size = 1_000_000
pulled_rows = []
usecols = ['subject_id', 'hadm_id', 'itemid', 'charttime', 'value', 'valuenum']

for chunk in pd.read_csv(
    raw_path + "chartevents.csv.gz",
    compression="gzip",
    usecols=usecols,
    chunksize=chunk_size
):
    filtered = chunk[
        chunk['itemid'].isin(all_relevant_itemids) &
        chunk['hadm_id'].isin(cohort_hadm_ids)
    ]
    if len(filtered) > 0:
        pulled_rows.append(filtered)

all_events = pd.concat(pulled_rows, ignore_index=True) if pulled_rows else pd.DataFrame(columns=usecols)
all_events['charttime'] = pd.to_datetime(all_events['charttime'])

print(f"Total relevant chartevents rows pulled (CAM-ICU features + RASS, cohort hadm_id only): {len(all_events)}")

cam_events = all_events[all_events['itemid'].isin(cam_icu_itemids)].copy()
rass_events = all_events[all_events['itemid'] == RASS_ITEMID].copy()

print(f"  CAM-ICU feature rows: {len(cam_events)}")
print(f"  RASS score rows: {len(rass_events)}")
print()

# ============================================================
# 1. Reconstruct CAM-ICU assessments: F1 AND F2 AND (F3 OR F4), same charttime
# ============================================================

cam_events['feature'] = cam_events['itemid'].map(feature_map)
cam_events['is_yes'] = cam_events['value'].astype(str).str.strip().str.startswith('Yes')

assessment_pivot = cam_events.pivot_table(
    index=['subject_id', 'hadm_id', 'charttime'],
    columns='feature',
    values='is_yes',
    aggfunc='max'
).reset_index()

for col in ['ms_change', 'inattention', 'disorganized_thinking', 'altered_loc']:
    if col not in assessment_pivot.columns:
        assessment_pivot[col] = False
assessment_pivot = assessment_pivot.fillna(False)

assessment_pivot['cam_icu_positive'] = (
    assessment_pivot['ms_change'] &
    assessment_pivot['inattention'] &
    (assessment_pivot['disorganized_thinking'] | assessment_pivot['altered_loc'])
)

positive_assessments = assessment_pivot[assessment_pivot['cam_icu_positive']].copy()

cam_icu_positive_subjects = set(positive_assessments['subject_id'])
n_cam_positive = len(cam_icu_positive_subjects)

print(f"Total CAM-ICU-positive patients (within cohort): {n_cam_positive}")
print(f"  (expecting close to ~3022 per your prior Notebook 1 run)")
print()

if n_cam_positive < 100:
    print("STOPPING: CAM-ICU-positive patient count is near zero — this indicates a join/logic")
    print("problem (mismatched itemid mapping, dtype mismatch on hadm_id, or a broken merge),")
    print("not a real data finding. Do not proceed to RASS windowing until this is fixed.")
else:
    # ============================================================
    # 2. For CAM-ICU-positive patients: pull RASS within +/-6h of ANY positive assessment
    # ============================================================

    rass_events_valid = rass_events.dropna(subset=['valuenum']).copy()

    windowed_rass_rows = []

    for subj_id, group in positive_assessments.groupby('subject_id'):
        subj_rass = rass_events_valid[rass_events_valid['subject_id'] == subj_id]
        if len(subj_rass) == 0:
            continue
        matched_indices = set()
        for ct in group['charttime']:
            window_start = ct - pd.Timedelta(hours=6)
            window_end = ct + pd.Timedelta(hours=6)
            in_window = subj_rass[
                (subj_rass['charttime'] >= window_start) & (subj_rass['charttime'] <= window_end)
            ]
            matched_indices.update(in_window.index)
        if matched_indices:
            windowed_rass_rows.append(subj_rass.loc[list(matched_indices)])

    cam_windowed_rass = pd.concat(windowed_rass_rows, ignore_index=True) if windowed_rass_rows else pd.DataFrame(columns=rass_events_valid.columns)

    print(f"RASS rows captured within +/-6h of CAM-ICU-positive assessments: {len(cam_windowed_rass)}")
    print(f"  Covering {cam_windowed_rass['subject_id'].nunique()} of {n_cam_positive} CAM-ICU-positive patients")
    print()

    # ============================================================
    # 3. ICD-delirium-only patients (no CAM-ICU-positive assessment): first 48h fallback
    # ============================================================

    delirium_final_codes = {'F05', '2930', '2931'}
    diagnoses_icd = pd.read_csv(raw_path + "diagnoses_icd.csv.gz", compression="gzip")
    diagnoses_icd['icd_code'] = diagnoses_icd['icd_code'].astype(str).str.strip()

    icd_delirium_hadm = set(
        diagnoses_icd.loc[
            diagnoses_icd['hadm_id'].isin(cohort_hadm_ids) &
            diagnoses_icd['icd_code'].isin(delirium_final_codes),
            'hadm_id'
        ]
    )
    icd_delirium_subjects = set(
        cohort_step1.loc[cohort_step1['hadm_id'].isin(icd_delirium_hadm), 'subject_id']
    )

    icd_fallback_subjects = icd_delirium_subjects - cam_icu_positive_subjects

    print(f"Patients with ICD delirium code: {len(icd_delirium_subjects)}")
    print(f"Patients routed to ICD 48h fallback (ICD-positive, CAM-ICU NOT positive): {len(icd_fallback_subjects)}")
    print()

    fallback_rass_rows = []
    for subj_id in icd_fallback_subjects:
        patient_row = cohort_step1[cohort_step1['subject_id'] == subj_id].iloc[0]
        window_start = patient_row['intime']
        window_end = patient_row['intime'] + pd.Timedelta(hours=48)
        subj_rass = rass_events_valid[
            (rass_events_valid['subject_id'] == subj_id) &
            (rass_events_valid['charttime'] >= window_start) &
            (rass_events_valid['charttime'] <= window_end)
        ]
        if len(subj_rass) > 0:
            fallback_rass_rows.append(subj_rass)

    icd_fallback_rass = pd.concat(fallback_rass_rows, ignore_index=True) if fallback_rass_rows else pd.DataFrame(columns=rass_events_valid.columns)

    print(f"RASS rows captured in ICD-fallback 48h windows: {len(icd_fallback_rass)}")
    print(f"  Covering {icd_fallback_rass['subject_id'].nunique()} of {len(icd_fallback_subjects)} fallback patients")
    print()

    # ============================================================
    # 4. Classify delirium_subtype_final
    # ============================================================

    def classify_subtype(sub_df):
        if len(sub_df) == 0:
            return 'Unknown'
        has_positive = (sub_df['valuenum'] >= 1).any()
        has_nonpositive = (sub_df['valuenum'] <= 0).any()
        if has_positive and has_nonpositive:
            return 'Mixed'
        elif has_positive:
            return 'Hyperactive'
        elif has_nonpositive:
            return 'Hypoactive'
        return 'Unknown'

    combined_windowed_rass = pd.concat([cam_windowed_rass, icd_fallback_rass], ignore_index=True)

    subtype_by_subject = combined_windowed_rass.groupby('subject_id').apply(classify_subtype).rename('delirium_subtype_final')
    subtype_full = subtype_by_subject.reindex(cohort_step1['subject_id']).fillna('Unknown')

    # ============================================================
    # 5. Print value_counts and routing counts
    # ============================================================

    print("=" * 55)
    print("delirium_subtype_final — value_counts()")
    print("=" * 55)
    print(subtype_full.value_counts())
    print()

    n_via_cam = cam_windowed_rass['subject_id'].nunique() if len(cam_windowed_rass) > 0 else 0
    n_via_icd_fallback = icd_fallback_rass['subject_id'].nunique() if len(icd_fallback_rass) > 0 else 0
    n_unknown_routing = len(cohort_step1) - n_via_cam - n_via_icd_fallback

    print("Routing counts:")
    print(f"  Via CAM-ICU +/-6h window: {n_via_cam}")
    print(f"  Via ICD 48h fallback:     {n_via_icd_fallback}")
    print(f"  Unknown (no RASS in either route): {n_unknown_routing}")
    print()

    # ============================================================
    # 6. Side-by-side comparison vs currently-saved whole-stay version
    # ============================================================

    old_counts = {'Mixed': 1878, 'Hypoactive': 1155, 'Hyperactive': 16, 'Unknown': 73}
    new_counts = subtype_full.value_counts().to_dict()

    comparison = pd.DataFrame({
        'Whole-stay (OLD)': pd.Series(old_counts),
        'Windowed (NEW)': pd.Series(new_counts)
    }).fillna(0).astype(int)

    print("=" * 55)
    print("SIDE-BY-SIDE COMPARISON")
    print("=" * 55)
    print(comparison)
    print()

    # ============================================================
    # 7. Decide whether to save
    # ============================================================

    hyperactive_count = new_counts.get('Hyperactive', 0)
    HYPERACTIVE_MIN_THRESHOLD = 20  # "still near-zero" cutoff

    if hyperactive_count >= HYPERACTIVE_MIN_THRESHOLD:
        cohort_step1['delirium_subtype'] = subtype_full.values
        cohort_step1.to_csv(processed_path + "cohort_step1.csv", index=False)
        print(f"Hyperactive count ({hyperactive_count}) clears the near-zero threshold — "
              f"REPLACING delirium_subtype column and saving cohort_step1.csv.")
        print(f"Final shape: {cohort_step1.shape}")
    else:
        print(f"Hyperactive count ({hyperactive_count}) is STILL near-zero even with proper windowing.")
        print("NOT overwriting delirium_subtype — this looks like a genuine data characteristic")
        print("(hyperactive delirium is rare/under-charted in this cohort), not a logic bug.")
        print("DECISION NEEDED: proceed with subtype modeling using only Hypoactive/Mixed/Unknown,")
        print("or document RASS-derived subtyping as infeasible and drop it from Notebook 3/4,")
        print("reporting it as a limitation instead.")

In [ ]:
import pandas as pd

processed_path = "../data/processed/"

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")

print(f"Current shape: {cohort_step1.shape}")
print(f"Current columns: {cohort_step1.columns.tolist()}")
print()

has_old = 'delirium_subtype' in cohort_step1.columns
has_new = 'delirium_subtype_final' in cohort_step1.columns

print(f"'delirium_subtype' present: {has_old}")
print(f"'delirium_subtype_final' present: {has_new}")
print()

if has_old and has_new:
    print("Both columns present — dropping old 'delirium_subtype' (whole-stay version), "
          "renaming 'delirium_subtype_final' -> 'delirium_subtype'.")
    cohort_step1 = cohort_step1.drop(columns=['delirium_subtype'])
    cohort_step1 = cohort_step1.rename(columns={'delirium_subtype_final': 'delirium_subtype'})

elif has_new and not has_old:
    print("Only 'delirium_subtype_final' present (no old column to drop) — "
          "renaming it to 'delirium_subtype'.")
    cohort_step1 = cohort_step1.rename(columns={'delirium_subtype_final': 'delirium_subtype'})

elif has_old and not has_new:
    print("Only 'delirium_subtype' present, no 'delirium_subtype_final' found — "
          "this means the windowed version was never saved as a separate column "
          "(likely already overwritten in-place in the previous step, or that step "
          "didn't clear the Hyperactive threshold and stopped before saving anything).")
    print("Nothing to merge/rename here. Leaving the file as-is.")

else:
    print("WARNING: Neither 'delirium_subtype' nor 'delirium_subtype_final' found in the file.")
    print("Nothing to do — check whether an earlier save step actually completed.")

print()
print(f"Columns after cleanup: {cohort_step1.columns.tolist()}")
print()

cohort_step1.to_csv(processed_path + "cohort_step1.csv", index=False)

print(f"Saved cohort_step1.csv")
print(f"Final shape: {cohort_step1.shape} (expected (3122, 12))")
print(f"Final columns: {cohort_step1.columns.tolist()}")

In [ ]:
import pandas as pd

raw_path = "../data/raw/"
processed_path = "../data/processed/"

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")

print(f"Loaded cohort_step1.csv: {cohort_step1.shape}")
print(f"Columns: {cohort_step1.columns.tolist()}")
print()

for col in ['delirium_subtype_x', 'delirium_subtype_y', 'delirium_subtype']:
    if col not in cohort_step1.columns:
        raise ValueError(f"Expected column '{col}' not found — check the file state before proceeding.")

# ============================================================
# 1. Print value_counts() for all three, clearly labeled
# ============================================================

print("=" * 55)
print("delirium_subtype_x — value_counts()")
print("=" * 55)
print(cohort_step1['delirium_subtype_x'].value_counts())
print()

print("=" * 55)
print("delirium_subtype_y — value_counts()")
print("=" * 55)
print(cohort_step1['delirium_subtype_y'].value_counts())
print()

print("=" * 55)
print("delirium_subtype — value_counts()")
print("=" * 55)
print(cohort_step1['delirium_subtype'].value_counts())
print()

# ============================================================
# 2. Check all three against the exact target
# ============================================================

target = {'Mixed': 1552, 'Hypoactive': 1429, 'Hyperactive': 76, 'Unknown': 65}

def matches_target(series, target):
    counts = series.value_counts().to_dict()
    return all(counts.get(k, 0) == v for k, v in target.items()) and \
           sum(counts.values()) == sum(target.values())

candidates = ['delirium_subtype_x', 'delirium_subtype_y', 'delirium_subtype']
matches = [col for col in candidates if matches_target(cohort_step1[col], target)]

print("=" * 55)
print(f"Target: {target}")
print("=" * 55)
for col in candidates:
    counts = cohort_step1[col].value_counts().to_dict()
    is_match = col in matches
    print(f"  {col}: {counts} -> {'MATCH' if is_match else 'no match'}")
print()
print(f"Column(s) matching exactly: {matches if matches else 'NONE'}")
print()

# ============================================================
# 3a / 3b. One or more exact matches found
# ============================================================

if len(matches) >= 1:
    keep_col = matches[0]
    drop_cols = [c for c in candidates if c != keep_col]

    if len(matches) > 1:
        print(f"Multiple columns matched exactly: {matches}. Keeping '{keep_col}', "
              f"dropping the rest as duplicates.")
    else:
        print(f"Exactly one column matched: '{keep_col}'. Dropping the other two.")

    cohort_step1 = cohort_step1.drop(columns=drop_cols)
    cohort_step1 = cohort_step1.rename(columns={keep_col: 'delirium_subtype'})

    cohort_step1.to_csv(processed_path + "cohort_step1.csv", index=False)

    print()
    print(f"Saved cohort_step1.csv")
    print(f"Final shape: {cohort_step1.shape} (expected (3122, 12))")
    print(f"Final columns: {cohort_step1.columns.tolist()}")
    print()
    shape_ok = cohort_step1.shape == (3122, 12)
    print(f"FINAL CONFIRMATION: cohort_step1.csv has exactly one delirium_subtype column "
          f"matching the target windowed logic, shape (3122, 12): {shape_ok}")

# ============================================================
# 3c. Zero matches — reconstruct from scratch, ignore all three existing columns
# ============================================================

else:
    print("NO column matched the target exactly. NOT SAVING the drop/rename above.")
    print("Reconstructing delirium_subtype from scratch, ignoring all three existing columns.")
    print()

    cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
    cohort_step1['outtime'] = pd.to_datetime(cohort_step1['outtime'])
    cohort_hadm_ids = set(cohort_step1['hadm_id'])

    feature_map = {
        228300: 'ms_change', 228337: 'ms_change', 229326: 'ms_change',
        228301: 'inattention', 228336: 'inattention', 229325: 'inattention',
        228303: 'disorganized_thinking', 228335: 'disorganized_thinking', 229324: 'disorganized_thinking',
        228334: 'altered_loc',
    }
    cam_icu_itemids = set(feature_map.keys())
    RASS_ITEMID = 228096
    all_relevant_itemids = cam_icu_itemids | {RASS_ITEMID}

    chunk_size = 1_000_000
    pulled_rows = []
    usecols = ['subject_id', 'hadm_id', 'itemid', 'charttime', 'value', 'valuenum']

    for chunk in pd.read_csv(
        raw_path + "chartevents.csv.gz",
        compression="gzip",
        usecols=usecols,
        chunksize=chunk_size
    ):
        filtered = chunk[
            chunk['itemid'].isin(all_relevant_itemids) &
            chunk['hadm_id'].isin(cohort_hadm_ids)
        ]
        if len(filtered) > 0:
            pulled_rows.append(filtered)

    all_events = pd.concat(pulled_rows, ignore_index=True) if pulled_rows else pd.DataFrame(columns=usecols)
    all_events['charttime'] = pd.to_datetime(all_events['charttime'])

    cam_events = all_events[all_events['itemid'].isin(cam_icu_itemids)].copy()
    rass_events = all_events[all_events['itemid'] == RASS_ITEMID].copy()
    rass_events_valid = rass_events.dropna(subset=['valuenum']).copy()

    print(f"CAM-ICU feature rows pulled: {len(cam_events)}")
    print(f"RASS score rows pulled: {len(rass_events)}")
    print()

    cam_events['feature'] = cam_events['itemid'].map(feature_map)
    cam_events['is_yes'] = cam_events['value'].astype(str).str.strip().str.startswith('Yes')

    assessment_pivot = cam_events.pivot_table(
        index=['subject_id', 'hadm_id', 'charttime'],
        columns='feature',
        values='is_yes',
        aggfunc='max'
    ).reset_index()

    for col in ['ms_change', 'inattention', 'disorganized_thinking', 'altered_loc']:
        if col not in assessment_pivot.columns:
            assessment_pivot[col] = False
    assessment_pivot = assessment_pivot.fillna(False)

    assessment_pivot['cam_icu_positive'] = (
        assessment_pivot['ms_change'] &
        assessment_pivot['inattention'] &
        (assessment_pivot['disorganized_thinking'] | assessment_pivot['altered_loc'])
    )

    positive_assessments = assessment_pivot[assessment_pivot['cam_icu_positive']].copy()
    cam_icu_positive_subjects = set(positive_assessments['subject_id'])
    print(f"CAM-ICU-positive patients: {len(cam_icu_positive_subjects)}")

    windowed_rass_rows = []
    for subj_id, group in positive_assessments.groupby('subject_id'):
        subj_rass = rass_events_valid[rass_events_valid['subject_id'] == subj_id]
        if len(subj_rass) == 0:
            continue
        matched_indices = set()
        for ct in group['charttime']:
            window_start = ct - pd.Timedelta(hours=6)
            window_end = ct + pd.Timedelta(hours=6)
            in_window = subj_rass[(subj_rass['charttime'] >= window_start) & (subj_rass['charttime'] <= window_end)]
            matched_indices.update(in_window.index)
        if matched_indices:
            windowed_rass_rows.append(subj_rass.loc[list(matched_indices)])

    cam_windowed_rass = pd.concat(windowed_rass_rows, ignore_index=True) if windowed_rass_rows else pd.DataFrame(columns=rass_events_valid.columns)
    print(f"RASS rows within +/-6h of CAM-ICU-positive assessments: {len(cam_windowed_rass)}")

    delirium_final_codes = {'F05', '2930', '2931'}
    diagnoses_icd = pd.read_csv(raw_path + "diagnoses_icd.csv.gz", compression="gzip")
    diagnoses_icd['icd_code'] = diagnoses_icd['icd_code'].astype(str).str.strip()

    icd_delirium_hadm = set(
        diagnoses_icd.loc[
            diagnoses_icd['hadm_id'].isin(cohort_hadm_ids) &
            diagnoses_icd['icd_code'].isin(delirium_final_codes),
            'hadm_id'
        ]
    )
    icd_delirium_subjects = set(
        cohort_step1.loc[cohort_step1['hadm_id'].isin(icd_delirium_hadm), 'subject_id']
    )
    icd_fallback_subjects = icd_delirium_subjects - cam_icu_positive_subjects
    print(f"ICD-delirium-only patients (fallback route): {len(icd_fallback_subjects)}")

    fallback_rass_rows = []
    for subj_id in icd_fallback_subjects:
        patient_row = cohort_step1[cohort_step1['subject_id'] == subj_id].iloc[0]
        window_start = patient_row['intime']
        window_end = patient_row['intime'] + pd.Timedelta(hours=48)
        subj_rass = rass_events_valid[
            (rass_events_valid['subject_id'] == subj_id) &
            (rass_events_valid['charttime'] >= window_start) &
            (rass_events_valid['charttime'] <= window_end)
        ]
        if len(subj_rass) > 0:
            fallback_rass_rows.append(subj_rass)

    icd_fallback_rass = pd.concat(fallback_rass_rows, ignore_index=True) if fallback_rass_rows else pd.DataFrame(columns=rass_events_valid.columns)
    print(f"RASS rows within ICD-fallback 48h windows: {len(icd_fallback_rass)}")
    print()

    def classify_subtype(sub_df):
        if len(sub_df) == 0:
            return 'Unknown'
        has_positive = (sub_df['valuenum'] >= 1).any()
        has_nonpositive = (sub_df['valuenum'] <= 0).any()
        if has_positive and has_nonpositive:
            return 'Mixed'
        elif has_positive:
            return 'Hyperactive'
        elif has_nonpositive:
            return 'Hypoactive'
        return 'Unknown'

    combined_windowed_rass = pd.concat([cam_windowed_rass, icd_fallback_rass], ignore_index=True)
    subtype_by_subject = combined_windowed_rass.groupby('subject_id').apply(classify_subtype).rename('delirium_subtype')
    subtype_full = subtype_by_subject.reindex(cohort_step1['subject_id']).fillna('Unknown')

    # Drop all three old columns, add the fresh one
    cohort_step1 = cohort_step1.drop(columns=['delirium_subtype_x', 'delirium_subtype_y', 'delirium_subtype'])
    cohort_step1['delirium_subtype'] = subtype_full.values

    print("=" * 55)
    print("Freshly reconstructed delirium_subtype — value_counts()")
    print("=" * 55)
    print(cohort_step1['delirium_subtype'].value_counts())
    print()

    cohort_step1.to_csv(processed_path + "cohort_step1.csv", index=False)

    print(f"Saved cohort_step1.csv")
    print(f"Final shape: {cohort_step1.shape} (expected (3122, 12))")
    print(f"Final columns: {cohort_step1.columns.tolist()}")
    print()
    shape_ok = cohort_step1.shape == (3122, 12)
    print(f"FINAL CONFIRMATION: cohort_step1.csv has exactly one delirium_subtype column "
          f"matching the intended windowed logic, shape (3122, 12): {shape_ok}")

In [ ]:
import pandas as pd, numpy as np, os
P = "../data/processed/"; RAW = "../data/raw/"

def find_raw(name):
    for r, _, fs in os.walk(RAW):
        if name in fs:
            return os.path.join(r, name)

pt = pd.read_csv(find_raw("patients.csv.gz"), usecols=["subject_id", "anchor_year", "anchor_year_group"])
co = pd.read_csv(P + "cohort_step1.csv", parse_dates=["intime"])
print("icu_admission_year == intime.year for all rows:", (co.intime.dt.year == co.icu_admission_year).all())

m = co[["subject_id", "stay_id", "hadm_id", "icu_admission_year", "mortality_28d"]].merge(
    pt, on="subject_id", how="left", validate="1:1")
assert m.anchor_year_group.notna().all()

g = m.anchor_year_group.str.extract(r"(\d{4})\s*-\s*(\d{4})").astype(int)
m["group_start"], m["group_end"] = g[0], g[1]
m["est_calendar_year"] = (((g[0] + g[1]) / 2) + (m.icu_admission_year - m.anchor_year)).round().astype(int)

print("\nEstimated real admission year (+/-1 yr):")
print(m.est_calendar_year.value_counts().sort_index().to_string())

print("\nCandidate cutoffs (train = est_calendar_year <= cutoff):")
rows = []
for c in range(2012, 2019):
    tr, te = m[m.est_calendar_year <= c], m[m.est_calendar_year > c]
    rows.append(dict(cutoff=c, n_train=len(tr), deaths_train=int(tr.mortality_28d.sum()),
                     n_test=len(te), deaths_test=int(te.mortality_28d.sum()),
                     mort_train=round(tr.mortality_28d.mean(), 3), mort_test=round(te.mortality_28d.mean(), 3)))
print(pd.DataFrame(rows).to_string(index=False))

print("\nEstimated year vs anchor group (sanity):")
print(pd.crosstab(m.est_calendar_year, m.anchor_year_group).to_string())

m.drop(columns=["mortality_28d"]).to_csv(P + "cohort_calendar_tags.csv", index=False)
print("\nSaved cohort_calendar_tags.csv (new file; cohort_step1.csv untouched)")

In [ ]:
import pandas as pd, numpy as np, os, re

P = "../data/processed/"; RAW = "../data/raw/"
def find_raw(name):
    for r, _, fs in os.walk(RAW):
        if name in fs:
            return os.path.join(r, name)
    raise FileNotFoundError(name)

# ------------------------------------------------------------
# Load current cohort (3,122) + exact in-hospital death time
# ------------------------------------------------------------
co = pd.read_csv(P + "cohort_step1.csv", parse_dates=["intime", "outtime"])
assert len(co) == 3122 and co.subject_id.is_unique and co.outtime.notna().all()
adm = pd.read_csv(find_raw("admissions.csv.gz"), usecols=["hadm_id", "deathtime"], parse_dates=["deathtime"])
co = co.merge(adm, on="hadm_id", how="left", validate="1:1")

# ============================================================
# PART 1 — PREDICTION-TIME ELIGIBILITY (alive AND still in ICU at hour 24)
# ============================================================
cut24 = co.intime + pd.Timedelta(hours=24)
died_24h = co.deathtime.notna() & (co.deathtime < cut24)
left_icu_24h = (~died_24h) & (co.outtime < cut24)

print("=" * 70, "\nPART 1: 24-HOUR ELIGIBILITY\n" + "=" * 70)
print(f"Start: {len(co)}")
print(f"Excluded - died within 24h of ICU admission: {died_24h.sum()} "
      f"(28d mortality in this group: {co.loc[died_24h,'mortality_28d'].mean():.2%})")
print(f"Excluded - left ICU alive within 24h:       {left_icu_24h.sum()} "
      f"(28d mortality in this group: {co.loc[left_icu_24h,'mortality_28d'].mean():.2%})")
print("\nCheck vs old immortal_time_flag (rows=old flag, cols=new died_24h):")
print(pd.crosstab(co.immortal_time_flag, died_24h))

elig = co.loc[~died_24h & ~left_icu_24h].copy()
print(f"\nEligible final cohort: {len(elig)}  | deaths: {int(elig.mortality_28d.sum())} "
      f"| 28d mortality: {elig.mortality_28d.mean():.2%}")

# ============================================================
# PART 2 — CANCER TYPE BY PRIMARY SITE (spread/unknown-site codes do not decide site)
# ============================================================
dd = pd.read_csv(find_raw("d_icd_diagnoses.csv.gz"), dtype={"icd_code": str})
dd["icd_code"] = dd.icd_code.str.strip()
dx = pd.read_csv(find_raw("diagnoses_icd.csv.gz"), dtype={"icd_code": str})
dx["icd_code"] = dx.icd_code.str.strip()

def icd9_cat(c):
    return int(c[:3]) if c[:3].isdigit() else None
def cancer_range(r):
    c = r.icd_code
    if r.icd_version == 9:
        k = icd9_cat(c)
        return k is not None and ((140 <= k <= 172) or (174 <= k <= 209))
    m = re.match(r"^C(\d{2})", c)
    return bool(m) and 0 <= int(m.group(1)) <= 96
def in_situ(r):
    c = r.icd_code
    if r.icd_version == 9:
        k = icd9_cat(c); return k is not None and 230 <= k <= 234
    return c[:2] == "D0"
excl_terms = "in remission|personal history|family history|screening|genetic susceptibility|follow-up|prophylactic|complicating pregnancy|complicating childbirth|complicating the puerperium"
keep = dd.apply(cancer_range, axis=1) & ~dd.long_title.str.contains(excl_terms, case=False, na=False) \
       & ~dd.icd_code.isin({"79582", "R971", "R9721", "D701"}) & ~dd.apply(in_situ, axis=1)
cancer_codes = dd.loc[keep, ["icd_code", "icd_version", "long_title"]]
print("\n" + "=" * 70, "\nPART 2: CANCER TYPE (PRIMARY SITE)\n" + "=" * 70)
print(f"Cancer code list size: {len(cancer_codes)} (expected 2052)")

rows = dx[dx.hadm_id.isin(elig.hadm_id)].merge(cancer_codes, on=["icd_code", "icd_version"], how="inner")
assert set(rows.hadm_id) == set(elig.hadm_id), "Every eligible admission must have >=1 cancer code"

def not_site_informative(code, ver):
    # secondary (spread) or unknown-primary codes
    if ver == 9:
        return code[:3] in {"196", "197", "198", "199"} or code.startswith("2097")
    return code[:3] in {"C77", "C78", "C79", "C80"}
def is_secondary(code, ver):
    if ver == 9:
        return code[:3] in {"196", "197", "198"} or code.startswith("2097")
    return code[:3] in {"C77", "C78", "C79"}

HEME = ['leukemia', 'lymphoma', 'myeloma', 'hodgkin', 'waldeyer', 'lymphoid', 'myeloid',
        'monocytic', 'erythroleukemia', 'plasma cell', 'mast cell leukemia']
def site_category(t):
    t = str(t).lower()
    if any(w in t for w in HEME): return 'Hematologic'
    if 'breast' in t: return 'Breast'
    if any(w in t for w in ['lung', 'bronchus', 'trachea', 'pleura', 'mediastinum']): return 'Respiratory/Thoracic'
    if any(w in t for w in ['lip', 'tongue', 'mouth', 'pharynx', 'larynx', 'salivary', 'nasal', 'sinus',
                             'nasopharynx', 'oropharynx', 'hypopharynx', 'tonsil', 'gum', 'palate']): return 'Head and Neck'
    if any(w in t for w in ['esophagus', 'stomach', 'colon', 'rectum', 'liver', 'pancrea', 'gallbladder',
                             'intestine', 'bile duct', 'anus', 'anal canal', 'peritoneum', 'digestive']): return 'Gastrointestinal'
    if any(w in t for w in ['kidney', 'bladder', 'prostate', 'testis', 'penis', 'ureter', 'urethra',
                             'urinary', 'renal pelvis']): return 'Genitourinary'
    if any(w in t for w in ['cervix', 'uterus', 'uterine', 'ovary', 'ovarian', 'vagina', 'vulva',
                             'fallopian', 'endometri']): return 'Gynecologic'
    if any(w in t for w in ['brain', 'meninges', 'spinal cord', 'cerebrum', 'cerebellum', 'nervous system',
                             'cranial nerve']): return 'CNS/Brain'
    if any(w in t for w in ['skin', 'melanoma', 'merkel cell']): return 'Skin'
    if any(w in t for w in ['thyroid', 'adrenal', 'pituitary', 'parathyroid', 'endocrine', 'pineal',
                             'carotid body']): return 'Endocrine'
    if any(w in t for w in ['bone', 'cartilage', 'connective', 'soft tissue']): return 'Bone/Soft Tissue'
    return 'Other/Unspecified'

rows["cat"] = rows.long_title.apply(site_category)
rows["uninformative"] = [not_site_informative(c, v) for c, v in zip(rows.icd_code, rows.icd_version)]
rows["secondary"] = [is_secondary(c, v) for c, v in zip(rows.icd_code, rows.icd_version)]

def patient_type(g):
    if (g.cat == 'Hematologic').any():
        return 'Hematologic'
    prim = g[~g.uninformative]
    if len(prim) == 0:
        return 'Other/Unspecified'                     # spread/unknown-site codes only
    cnt = prim.groupby('cat').agg(n=('cat', 'size'), first_seq=('seq_num', 'min'))
    cnt = cnt.sort_values(['n', 'first_seq'], ascending=[False, True])
    return cnt.index[0]                                # most frequent primary site; tie -> earliest-listed code

ptype = rows.groupby('hadm_id')[['cat', 'uninformative', 'seq_num']].apply(patient_type).rename('cancer_type_raw')
metas = rows.groupby('hadm_id')['secondary'].any().rename('metastatic_code_flag')   # descriptive (Table 1) only
elig = elig.merge(ptype, on='hadm_id', how='left', validate='1:1').merge(metas, on='hadm_id', how='left', validate='1:1')

raw_counts = elig.cancer_type_raw.value_counts()
small = raw_counts[raw_counts < 100].index.tolist()
elig['cancer_type'] = elig.cancer_type_raw.where(~elig.cancer_type_raw.isin(small), 'Other/Unspecified')
print("\nPrimary-site counts before merging small groups:"); print(raw_counts.to_string())
print(f"\nMerged into Other/Unspecified (<100 patients): {small}")
print("\nFinal cancer_type counts:"); print(elig.cancer_type.value_counts().to_string())
print("\nOld vs new cancer_type (rows=old, cols=new):")
print(pd.crosstab(elig['cancer_type_x'] if 'cancer_type_x' in elig else co.set_index('hadm_id').loc[elig.hadm_id, 'cancer_type'].values,
                  elig.cancer_type).to_string())

# ============================================================
# PART 3 — CALENDAR ERA (temporal validation tag)
# ============================================================
pt = pd.read_csv(find_raw("patients.csv.gz"), usecols=["subject_id", "gender", "anchor_year", "anchor_year_group"])
elig = elig.merge(pt, on="subject_id", how="left", validate="1:1")
g = elig.anchor_year_group.str.extract(r"(\d{4})\s*-\s*(\d{4})").astype(int)
elig["est_calendar_year"] = ((g[0] + g[1]) / 2 + (elig.intime.dt.year - elig.anchor_year)).round().clip(2008, 2022).astype(int)
TEMPORAL_CUTOFF = 2017   # pre-specified on era balance only, before any model result
elig["temporal_set"] = np.where(elig.est_calendar_year <= TEMPORAL_CUTOFF, "development_era", "validation_era")
print("\n" + "=" * 70, "\nPART 3: CALENDAR ERA\n" + "=" * 70)
print(pd.crosstab(elig.temporal_set, elig.mortality_28d, margins=True).to_string())

# ============================================================
# SAVE (new file; cohort_step1.csv untouched)
# ============================================================
out_cols = ["subject_id", "hadm_id", "stay_id", "intime", "outtime", "age_at_admission", "gender",
            "mortality_28d", "cancer_type", "metastatic_code_flag",
            "anchor_year_group", "est_calendar_year", "temporal_set"]
final = elig[out_cols].copy()
assert final.subject_id.is_unique and final[out_cols].drop(columns=["metastatic_code_flag"]).notna().all().all()
assert (final.outtime >= final.intime + pd.Timedelta(hours=24)).all()
final.to_csv(P + "cohort_final.csv", index=False)

attrition = pd.DataFrame([
    ("Cancer-positive admissions (2,052 ICD codes)", 74503),
    ("Delirium-positive admissions (ICD or CAM-ICU 4-feature)", 12309),
    ("Cancer AND delirium admissions", 4361),
    ("Unique patients", 3950),
    ("Patients with an ICU stay (first stay, age>=18)", 3123),
    ("After excluding missing ICU discharge time", 3122),
    ("After excluding death within 24h of ICU admission", 3122 - int(died_24h.sum())),
    ("After excluding ICU discharge within 24h (FINAL)", len(final)),
], columns=["stage", "n"])
attrition.to_csv(P + "attrition_counts_final.csv", index=False)
print("\nSaved cohort_final.csv", final.shape, "and attrition_counts_final.csv")
print(attrition.to_string(index=False))

In [ ]:
import pandas as pd, os
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

P = "../data/processed/"; FIG = "../figures/"
os.makedirs(FIG, exist_ok=True)

att = pd.read_csv(P + "attrition_counts_final.csv")
n = dict(zip(range(len(att)), att.n))
co = pd.read_csv(P + "cohort_final.csv")
dev = co[co.temporal_set == "development_era"]; val = co[co.temporal_set == "validation_era"]
f = lambda x: f"{x:,}"

main = [
    f"Admissions with ≥1 cancer ICD code\n(2,052 ICD-9/10 codes; MIMIC-IV v3.1)\nn = {f(n[0])} admissions",
    f"Admissions with cancer AND delirium\n(ICD F05/293.0/293.1 or positive CAM-ICU*)\nn = {f(n[2])} admissions ({f(n[3])} patients)",
    f"First ICU stay, age ≥18 years\nn = {f(n[4])} patients",
    f"Valid ICU discharge time\nn = {f(n[5])}",
    f"Alive at 24 h after ICU admission\nn = {f(n[6])}",
    f"FINAL COHORT (in ICU at 24 h)\nn = {f(n[7])}\n28-day deaths: {f(int(co.mortality_28d.sum()))} ({co.mortality_28d.mean():.1%})",
]
excl = [
    f"Excluded: no delirium evidence\nn = {f(n[0] - n[2])} admissions",
    f"Excluded: no ICU stay in a\nqualifying admission\nn = {f(n[3] - n[4])} patients",
    f"Excluded: missing ICU\ndischarge time\nn = {f(n[4] - n[5])}",
    f"Excluded: died within 24 h\nof ICU admission\nn = {f(n[5] - n[6])}",
    f"Excluded: discharged from ICU\nalive within 24 h\nn = {f(n[6] - n[7])}",
]

fig, ax = plt.subplots(figsize=(11, 15)); ax.set_xlim(0, 11); ax.set_ylim(0, 16); ax.axis("off")
def box(x, y, w, h, t, bold=False, fc="white"):
    ax.add_patch(FancyBboxPatch((x - w/2, y - h/2), w, h, boxstyle="round,pad=0.08", fc=fc, ec="black", lw=1.2))
    ax.text(x, y, t, ha="center", va="center", fontsize=10, fontweight="bold" if bold else "normal")
def arrow(x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", lw=1.2))

ys = [15, 12.6, 10.3, 8.2, 6.1, 3.9]
for i, (y, t) in enumerate(zip(ys, main)):
    box(3.6, y, 5.6, 1.4, t, bold=(i == len(main) - 1), fc="#eef4fb" if i == len(main) - 1 else "white")
    if i > 0:
        arrow(3.6, ys[i-1] - 0.75, 3.6, y + 0.75)
        ymid = (ys[i-1] + y) / 2
        ax.plot([3.6, 7.2], [ymid, ymid], color="black", lw=1)
        box(8.9, ymid, 3.4, 1.15, excl[i-1], fc="#fbeeee")

box(2.0, 1.4, 3.6, 1.3, f"Development era (≤2017)\nn = {f(len(dev))}\ndeaths {f(int(dev.mortality_28d.sum()))} ({dev.mortality_28d.mean():.1%})")
box(7.2, 1.4, 3.6, 1.3, f"Temporal validation era (>2017)\nn = {f(len(val))}\ndeaths {f(int(val.mortality_28d.sum()))} ({val.mortality_28d.mean():.1%})")
arrow(3.0, ys[-1] - 0.75, 2.0, 2.1); arrow(4.2, ys[-1] - 0.75, 7.2, 2.1)
ax.text(5.5, 0.15, "*CAM-ICU positive = acute change AND inattention AND (disorganized thinking OR altered "
        "consciousness) at the same assessment; evaluated within cancer-positive admissions.\n"
        "Random 80/20 stratified train/test split of the final cohort is performed in the modelling stage.",
        ha="center", va="bottom", fontsize=8, style="italic")
plt.tight_layout()
plt.savefig(FIG + "attrition_diagram_final.png", dpi=300, bbox_inches="tight")
plt.savefig(FIG + "attrition_diagram_final.pdf", bbox_inches="tight")
plt.show()
print("Saved attrition_diagram_final.png / .pdf")

In [ ]:
import os, shutil
os.makedirs("../figures_v2", exist_ok=True)
for ext in ["png", "pdf"]:
    shutil.copy(f"../figures/attrition_diagram_final.{ext}", f"../figures_v2/attrition_diagram_final.{ext}")
print(sorted(os.listdir("../figures_v2")))

In [ ]:
# ========= STEP 1b (Notebook 1, last cell): WHEN was delirium documented? =========
# Re-derives the delirium inclusion rule from RAW files and records its timing.
# Output is a cohort descriptor for Table 1 + a sensitivity analysis in NB4.
# It is NOT a model feature. No outcome column is read.
import pandas as pd, numpy as np, os, time, shutil

RAW = "../data/raw/"; P1 = "../data/processed/"; P2 = "../data/processed_v2/"
def find_raw(name):
    for r, _, fs in os.walk(RAW):
        if name in fs:
            return os.path.join(r, name)
    raise FileNotFoundError(name)

co = pd.read_csv(P2 + "cohort_final.csv", usecols=["stay_id", "hadm_id", "intime"],
                 parse_dates=["intime"])
N = len(co)
hadms = set(co.hadm_id)
t0 = co.set_index("hadm_id").intime

# ---------------- A. ICD delirium codes (same exact set as the inclusion rule) ----------------
DEL_ICD = {"F05", "2930", "2931"}
dx = pd.read_csv(find_raw("diagnoses_icd.csv.gz"), dtype={"icd_code": str},
                 usecols=["hadm_id", "icd_code"])
dx = dx[dx.hadm_id.isin(hadms)]
icd_hadm = set(dx[dx.icd_code.str.strip().isin(DEL_ICD)].hadm_id)

# ---------------- B. CAM-ICU 4-feature rule (same rule as the inclusion step) ----------------
FEAT = {228300: "ms_change", 228337: "ms_change", 229326: "ms_change",
        228301: "inattention", 228336: "inattention", 229325: "inattention",
        228303: "disorganized", 228335: "disorganized", 229324: "disorganized",
        228334: "altered_loc"}
t = time.time()
parts = []
for ch in pd.read_csv(find_raw("chartevents.csv.gz"), chunksize=2_000_000,
                      usecols=["hadm_id", "charttime", "itemid", "value"],
                      dtype={"value": str}):
    ch = ch[ch.itemid.isin(FEAT) & ch.hadm_id.isin(hadms)]
    if len(ch):
        parts.append(ch)
cam = pd.concat(parts, ignore_index=True)
print(f"CAM-ICU rows for cohort admissions: {len(cam):,} ({(time.time() - t) / 60:.1f} min)")
cam["charttime"] = pd.to_datetime(cam.charttime)
cam["feature"] = cam.itemid.map(FEAT)
cam["yes"] = cam.value.astype(str).str.strip().str.startswith("Yes")
a = cam.pivot_table(index=["hadm_id", "charttime"], columns="feature", values="yes",
                    aggfunc="max").reset_index()
for c in ["ms_change", "inattention", "disorganized", "altered_loc"]:
    if c not in a:
        a[c] = False
    a[c] = a[c].astype("boolean").fillna(False).astype(bool)
a["positive"] = a.ms_change & a.inattention & (a.disorganized | a.altered_loc)
a["hours"] = (a.charttime - a.hadm_id.map(t0)).dt.total_seconds() / 3600

pos = a[a.positive]
first_pos = pos.groupby("hadm_id").hours.min()
assessed_24 = set(a[a.hours <= 24].hadm_id)

# ---------------- C. per-patient timing table ----------------
T = co[["stay_id", "hadm_id"]].copy()
T["delirium_icd"] = T.hadm_id.isin(icd_hadm).astype(int)
T["cam_positive_any"] = T.hadm_id.isin(set(pos.hadm_id)).astype(int)
T["cam_first_positive_h"] = T.hadm_id.map(first_pos)
T["cam_positive_by_24h"] = (T.cam_first_positive_h <= 24).astype(int)
T["cam_assessed_by_24h"] = T.hadm_id.isin(assessed_24).astype(int)
T["delirium_route"] = np.select(
    [(T.delirium_icd == 1) & (T.cam_positive_any == 1), T.delirium_icd == 1,
     T.cam_positive_any == 1], ["ICD + CAM-ICU", "ICD only", "CAM-ICU only"], "NONE")

# every patient must meet the inclusion rule again from raw data
n_none = int((T.delirium_route == "NONE").sum())
assert n_none == 0, f"{n_none} patients have no delirium evidence - inclusion rule broken"
assert len(T) == N and T.stay_id.is_unique
T.to_csv(P2 + "cohort_delirium_timing.csv", index=False)

print("Saved processed_v2/cohort_delirium_timing.csv", T.shape)
print("\nInclusion rule re-derived from raw data: every patient has ICD or CAM-ICU evidence")
print("\nDelirium route:", T.delirium_route.value_counts().to_dict())
print("CAM-ICU positive at or before 24h:", int(T.cam_positive_by_24h.sum()))
print("CAM-ICU positive only after 24h:",
      int(((T.cam_positive_any == 1) & (T.cam_positive_by_24h == 0)).sum()))
print("Any CAM-ICU assessment by 24h:", int(T.cam_assessed_by_24h.sum()))
q = T.cam_first_positive_h.describe(percentiles=[.25, .5, .75])
print("Hours from ICU admission to first positive CAM-ICU:",
      f"median {q['50%']:.1f} (IQR {q['25%']:.1f}-{q['75%']:.1f})")

# ---------------- D. supplement tables into the final v2 folder ----------------
# These were made by the verified NB1 steps (code list unchanged, attrition from Step 1).
for f in ["attrition_counts_final.csv", "icd_code_appendix.csv", "icd_code_exclusions.csv"]:
    if os.path.exists(P1 + f):
        shutil.copy(P1 + f, P2 + f)
        print("Copied to processed_v2:", f, pd.read_csv(P2 + f).shape)
    else:
        print("NOT FOUND (tell me):", f)